# Sesión 4 — Spark como motor de procesamiento distribuido

Esta sesión es el siguiente paso después de S1, S2 y S3. En S1 se
estudiaron HDFS, YARN y MapReduce; en S2 se exploraron los Parquet de TPC-DS
SF3 como ficheros, sin catálogo; en S3 se comprobó que esos mismos Parquet
pueden vivir en un almacenamiento compatible con S3. Ahora se incorpora un
motor de procesamiento distribuido capaz de leer esos ficheros y ejecutar el
trabajo sobre los DataNodes a través de YARN.

La pregunta central de esta sesión es:

> ¿Cómo pasa una aplicación Python de leer datos desde una ruta a construir y
> ejecutar un trabajo distribuido sobre los DataNodes?

La figura sitúa la sesión en la arquitectura física del laboratorio: cada círculo es un contenedor Docker, y los puntos de color de su interior son los servicios que ejecuta; a la derecha están los servicios, con el color de la sesión que los introduce y una flecha hacia el clúster. En color está lo que ya existe al empezar la sesión y sombreado, lo que llegará en sesiones posteriores; resaltado, lo que esta sesión crea, modifica o usa. S4 añade Spark: el driver vive en `namenode` y los ejecutores, que YARN reparte por los DataNodes, leen los Parquet de `/datalake/raw/tpcds`.

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/arquitectura-s4.svg" alt="Arquitectura del laboratorio al empezar la sesión 4: se añade Spark, con su driver y sus ejecutores sobre YARN, que lee el data lake en HDFS" width="1100" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

Los datos de entrada siguen siendo los de TPC-DS SF3. Se conserva el
materializado original de `/datalake/raw/tpcds` y se evita modificarlo
durante los ejercicios: las transformaciones de Spark escribirán sus
salidas en rutas de trabajo separadas, bajo `/user/luser/s4`.

> **Paquetes de Python de esta sesión.** S4 usa `pyspark` para leer y
> transformar Parquet con Spark, tanto localmente como en YARN. Todos los
> notebooks de S2-S8 instalan la lista común publicada como
> `entorno/requirements.txt`, que reúne los paquetes utilizados a lo largo del
> curso. La imagen de `namenode` ya trae esa lista en `/opt/tcdm/venv`, el
> intérprete del kernel; lo normal es que `%pip` confirme que están.

## Actualizar el material antes de empezar

Antes de empezar, desde la raíz de la distribución y en una terminal de tu
equipo:

```bash
make -C entorno actualizar
```

La orden trae la última versión publicada del material. No pisa tu trabajo: si
has modificado un fichero que también cambia en la versión nueva, se aparta
como copia `<fichero>.mio-<fecha>-<hora>` y el original pasa a ser el
publicado. Lo que la versión nueva no cambia se queda como está.


### Consultar el esquema de TPC-DS

Como en S2, `%esquema` muestra debajo de la celda en la que lo escribas la
descripción de una tabla o de una columna de TPC-DS, tomada del apéndice de
`s2.ipynb`:

| Celda | Qué muestra |
| --- | --- |
| `%esquema` | Las 18 tablas, con su papel y su descripción |
| `%esquema web_sales` | La tabla completa: su descripción, su diagrama entidad-relación si es una tabla de hechos, sus columnas y dos filas de muestra |
| `%esquema web_sales date` | Las columnas de `web_sales` cuyo nombre contiene `date` |
| `%esquema ws_sold_timestamp` | Esa columna y ninguna otra |
| `%esquema item_sk` | Todas las columnas `item_sk`, tabla a tabla: quién apunta a `item` |
| `%esquema web_sales customer` | Varias tablas seguidas, por ejemplo las de un `JOIN` |

Para pedir varias cosas a la vez también sirve una celda `%%esquema` con una
tabla, columna o filtro por línea. Lo encontrarás la primera vez que la
sesión usa cada tabla; añade una celda de éstas donde te haga falta y bórrala
después.

`%schema` y `%%schema` son los mismos comandos con su nombre en inglés:
usa el que prefieras.

La celda siguiente lo prepara: ejecútala una vez por sesión de kernel.
Descarga `entorno/python_utils/tcdm_esquema.py` del repositorio público
`dsevilla/tcdm-public` (rama `26-27`), porque el kernel se ejecuta en
`namenode`, que no ve tu copia de la distribución. Si el fichero ya está en
el directorio de trabajo del kernel, se conserva; bórralo para traer una
versión nueva. Si hay que descargarlo, `namenode` necesita acceso a Internet.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

SCHEMA_MODULE: Path = Path("tcdm_esquema.py")
schema_module_url: str = (
    "https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/"
    "entorno/python_utils/tcdm_esquema.py"
)

if not SCHEMA_MODULE.exists():
    request: Request = Request(schema_module_url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        SCHEMA_MODULE.write_bytes(response.read())

%load_ext tcdm_esquema

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:linear-gradient(135deg,#0c304d,#174f7a 62%,#a54467);color:#ffffff;border:1px solid transparent;display:flex;flex-direction:column;justify-content:center;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="margin:0 auto;max-width:44ch;text-align:left"><h1 align="center" style="color:#ffffff;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;text-align:center;margin:0 0 22px">Sesión 4: Spark como motor de procesamiento distribuido</h1><h2 align="center" style="color:#dceaf4;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:30px;text-align:center;margin:18px 0 8px">Cómo vamos a recorrer la sesión</h2><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Spark es a día de hoy el motor de procesamiento distribuido más popular en el mundo del Big Data</li><li style='margin:10px 0;list-style-position:outside'>Un motor nuevo sobre los mismos datos: los Parquet de TPC-DS SF3</li><li style='margin:10px 0;list-style-position:outside'>Se basa en DataFrames: transformaciones perezosas, acciones y planes de ejecución</li><li style='margin:10px 0;list-style-position:outside'>Las mismas preguntas de negocio de S2, ahora con otro motor</li></ul></div><hr color="#174f7a" size="8" noshade style="display:none"></div>

## Capas que se deben distinguir

En esta sesión aparecen varias palabras que se reutilizarán más adelante y
que conviene separar desde el principio:

| Concepto | Significado en esta sesión |
| --- | --- |
| HDFS o S3 | Sistema de almacenamiento que contiene los datos |
| Parquet | Formato columnar de los ficheros |
| Spark | Motor que construye y ejecuta el procesamiento |
| YARN | Gestor que asigna recursos a la aplicación Spark |
| Partición de Spark | Parte lógica del trabajo que puede ejecutar una tarea |
| Partición Hive | Organización física de datos por valores de columnas; se verá en S5 |
| Partición de ventana | Grupo de filas sobre el que opera una función de ventana (`Window.partitionBy`); se verá en las funciones de ventana |
| Catálogo | Servicio que da nombres y metadatos a las tablas; se verá en S6 y S7 |

Una partición de ejecución de Spark **no** es un directorio de partición
Hive. Tampoco es un bloque HDFS, ni un fichero Parquet, ni el grupo de una función de ventana. Que compartan el
nombre no significa que sean la misma unidad; a lo largo de esta sesión se
insistirá varias veces en esa distinción.

## Objetivos de la sesión

Al terminar esta sesión deberías poder:

- crear una `SparkSession` y ejecutar una aplicación PySpark;
- leer Parquet desde una ruta HDFS y examinar su esquema;
- construir DataFrames con `select`, `filter`, `withColumn`, `join` y
  `groupBy`;
- distinguir transformaciones de acciones;
- explicar que Spark construye un plan antes de ejecutar el trabajo;
- observar el plan lógico y físico con `explain`;
- diferenciar el driver, los ejecutores y el gestor de recursos;
- comparar los modos `local` y `yarn`;
- comprender qué son las particiones de ejecución de Spark;
- decidir cuándo compensa cachear un DataFrame reutilizado y liberarlo al
  terminar;
- comprobar qué ejecutores remotos y qué DataNodes realizan trabajo;
- escribir resultados Parquet sin confundir todavía una salida de Spark con
  una tabla registrada;
- utilizar vistas temporales para introducir Spark SQL sin un catálogo
  permanente;
- conservar consultas reproducibles y explicar la métrica que calcula cada
  ejemplo (líneas de venta frente a pedidos distintos, por ejemplo).

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #b3701a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#b3701a" size="8" noshade style="display:none"><div style="color:#b3701a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#b3701a" style="color:inherit"><b>Evaluación de la sesión</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cómo se evalúa esta sesión</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Reunión individual de unos 5 minutos con el profesor</li><li style='margin:10px 0;list-style-position:outside'>Se muestran en vivo, en tu ordenador, los tres ejercicios con el dataset resueltos y ejecutados en tu notebook</li><li style='margin:10px 0;list-style-position:outside'>Se muestra en vivo, en tu ordenador: la aplicación en YARN, el informe de hosts que procesaron particiones y el resultado de negocio idéntico en local y sobre YARN</li><li style='margin:10px 0;list-style-position:outside'>Se entrega también una memoria breve (una o dos páginas)</li><li style='margin:10px 0;list-style-position:outside'>No hace falta memorizar órdenes: sí explicar qué papel juega cada pieza</li></ul><hr color="#b3701a" size="2" noshade style="display:none"></div>

## Antes de empezar

> **Dónde se ejecuta este notebook.** Igual que en S2 y S3, Jupyter se
> ejecuta directamente dentro del contenedor `namenode`, como `luser`, con
> acceso de red directo a HDFS y YARN. Las celdas ejecutables hablan
> directamente con el clúster. Las órdenes que necesitan
> el Docker del host — arrancar o detener contenedores, `make` — se
> muestran como texto para ejecutarlas en una terminal de tu equipo, nunca
> como celdas de este notebook.

Esta sesión asume que el clúster de S1 ya está en marcha y que S2 generó
los Parquet de TPC-DS SF3 en `/datalake/raw/tpcds`. Si no es así, desde una
terminal de tu equipo situada en la raíz de la distribución de sesiones:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-host-hadoop-up.gif" alt="Animación: la orden make -C entorno hadoop-up se teclea en una terminal de tu equipo (host)" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

```bash
cd ~/tcdm-public
make -C entorno hadoop-up
```

Este notebook se ejecuta en el Jupyter de `namenode`, que no arranca solo.
Si el Jupyter de la sesión anterior sigue en marcha (su terminal está abierta),
no lo lances otra vez: un segundo servidor usaría otro puerto, que no se
publica en tu equipo. Salta al paso de conectar el notebook. Si no, como en S2,
arráncalo desde **otra terminal de tu equipo** y déjala abierta mientras
trabajes:

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-namenode-jupyter.gif" alt="Animación: docker exec -it namenode bash se teclea en tu equipo y abre una terminal dentro del contenedor namenode, donde se ejecutan su - luser y jupyter lab" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

```bash
# en tu equipo: entra en namenode
docker exec -it namenode bash
# ya dentro de namenode, como luser: Jupyter con el token fijo «tcdm»
su - luser
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --IdentityProvider.token=tcdm
```

La primera orden es la única que se ejecuta en tu equipo; las otras dos, ya
dentro del contenedor, arrancan Jupyter con el Python del curso
(`make -C entorno jupyter` hace lo mismo en una sola orden). En Visual
Studio Code, conecta este notebook con **Select Kernel → Select Another
Kernel → Existing Jupyter Server**, usando la dirección
`http://127.0.0.1:8888/lab?token=tcdm` y eligiendo el kernel **Python 3
(ipykernel)**. El token es siempre `tcdm`.

Si algo no funciona —el notebook no conecta, una celda `!hdfs` no
encuentra la orden, un servicio no responde—, consulta «Solución de
problemas» en `entorno/README.md`.

Antes de instalar nada, comprueba que los datos de entrada siguen donde S2
los dejó:

In [ ]:
!hdfs dfs -chmod -R o+rX /datalake /warehouse
!hdfs dfs -ls -R -h /datalake/raw/tpcds

Puedes verlos también en el navegador:

- <http://localhost:9870/explorer.html#/datalake>

Si el directorio no existe, revisa S2 antes de continuar: esta sesión no
regenera TPC-DS.

El progreso y los planes se observan con `.show()`, `.explain()` y la CLI
de `yarn`. El ResourceManager (`:8088`) y el
Timeline Server (`:8188`), ya conocidos desde S1, siguen disponibles para
mirar el estado de las aplicaciones YARN en el navegador. A ellos se suma
la interfaz web del propio Spark (`:4040`), que se presenta en cuanto exista
la `SparkSession`.

## Instalar PySpark en este notebook

La celda siguiente instala la lista común publicada en `entorno/requirements.txt`, directamente desde su URL HTTPS. `pip` admite una URL en `-r`, así que no hace falta escribir una copia local del fichero.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/dsevilla/tcdm-public/26-27/entorno/requirements.txt

In [ ]:
import sys

import pyspark

print(f"PySpark {pyspark.__version__} con Python {sys.version.split()[0]}.")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">El motor Spark</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Por qué Spark aparece como respuesta al procesamiento MapReduce</li><li style='margin:10px 0;list-style-position:outside'>Driver, ejecutores y gestor de recursos: tres papeles en cualquier modo</li><li style='margin:10px 0;list-style-position:outside'>Job, stage, task y partición: las unidades que verás en los planes</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## El motor Spark

**Apache Spark nació en 2009** en el AMPLab de UC Berkeley, motivado por lo
costoso que resultaba encadenar trabajos iterativos con el modelo MapReduce
que ya conoces de S1: cada paso de un pipeline MapReduce vuelve a leer y
escribir HDFS, aunque los datos quepan en memoria. Se hizo open source en
2010 y pasó a la Apache Software Foundation en 2013. Spark generaliza el
modelo de MapReduce: sigue dividiendo el trabajo en tareas sobre particiones
de datos, pero encadena esas tareas en un grafo de ejecución y puede
mantener resultados intermedios en memoria entre una etapa y la siguiente,
en lugar de pasar obligatoriamente por HDFS entre cada paso.

Spark ofrece varios módulos sobre un mismo motor central: SQL/DataFrames,
streaming estructurado, una biblioteca de aprendizaje automático (MLlib) y
procesamiento de grafos. Esta sesión se queda únicamente en SQL/DataFrames,
que es el módulo relevante para un data lake de ficheros Parquet.

### API estructurada frente a RDDs

Spark ofrece dos niveles de API:

- la **API estructurada** (DataFrames y, en Scala/Java, Datasets tipados),
  que describe *qué* se quiere calcular sobre columnas con nombre y deja
  que decidan *cómo* ejecutarlo el optimizador Catalyst (que reescribe el
  plan: descarta las columnas que no se usan, adelanta los filtros hacia
  la lectura...) y el motor de ejecución Tungsten (que ejecuta ese plan
  sobre una representación binaria compacta de las filas);
- la **API de bajo nivel** (RDDs, *Resilient Distributed Datasets*), la
  interfaz original de Spark 1.x: colecciones distribuidas de objetos
  Python arbitrarios, sin optimizador de por medio.

Los RDDs siguen existiendo y en algún punto muy concreto de esta sesión
—al final, en la sección sobre YARN— se usará uno para hacer visible en qué
host se ha ejecutado cada tarea, porque es la única forma directa de leer
`socket.gethostname()` dentro de cada partición. Fuera de ese caso muy
puntual, todo el notebook usa exclusivamente DataFrames: es la API
recomendada.

### Driver, ejecutores y gestor de recursos

Tres papeles se repiten en cualquier ejecución de Spark, en modo local o sobre YARN:

- el **driver** crea la `SparkSession`, construye el plan lógico y físico
  de cada consulta y coordina la ejecución;
- los **ejecutores** son los procesos que ejecutan las tareas sobre las
  particiones de datos y pueden mantener resultados en memoria;
- el **gestor de recursos** decide dónde se ejecutan el driver y los
  ejecutores. En modo `local[N]` ese gestor no existe realmente: todo vive
  en un único proceso JVM con `N` hilos simulando ejecutores. Sobre YARN, el
  gestor de recursos es el mismo ResourceManager que ya conoces de S1: el
  driver pide un *ApplicationMaster* y unos contenedores de ejecutor, y los
  NodeManagers de los DataNodes los conceden.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Términos de los planes de ejecución</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><strong>Partición</strong>: grupo de filas que se procesa de una vez</li><li style='margin:10px 0;list-style-position:outside'><strong>Task</strong>: la cadena de operaciones aplicada a una partición</li><li style='margin:10px 0;list-style-position:outside'><strong>Stage</strong>: tareas que se ejecutan sin redistribuir datos (sin <em>shuffle</em>)</li><li style='margin:10px 0;list-style-position:outside'><strong>Job</strong>: las <em>stages</em> que dispara una acción</li><li style='margin:10px 0;list-style-position:outside'><strong>Shuffle</strong>: reparto de filas por clave entre particiones; pasa por disco y red y separa una <em>stage</em> de la siguiente</li><li style='margin:10px 0;list-style-position:outside'><strong>Dependencia estrecha</strong> (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>select</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>filter</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>withColumn</code>): cada partición de salida sale de una sola de entrada</li><li style='margin:10px 0;list-style-position:outside'><strong>Dependencia ancha</strong> (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>groupBy</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>orderBy</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>join</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>distinct</code>): necesita varias particiones de entrada, y por tanto un <em>shuffle</em></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

### Términos que reaparecerán en los planes de ejecución

- **Partición**: un trozo del DataFrame —un grupo de filas— que se procesa
  de una vez. Si `web_sales` se lee en tres particiones, Spark tiene tres
  trozos que repartir entre los ejecutores.
- **Task**: la unidad de trabajo: aplicar una cadena de operaciones a una
  partición. Tres particiones, tres tareas por etapa.
- **Stage**: grupo de tareas que se pueden ejecutar sin necesidad de
  redistribuir datos entre ellas (sin *shuffle*).
- **Job**: conjunto de *stages* que dispara una acción.
- **Shuffle**: reparto de las filas entre particiones según el valor de
  una clave (por ejemplo, que todas las líneas de un mismo cliente acaben
  en la misma partición para poder sumarlas). Las filas se escriben a disco
  y viajan entre ejecutores: por eso es la operación cara de Spark y la que
  separa una *stage* de la siguiente.
- **Dependencia estrecha** (*narrow*): cada partición de salida depende de
  una única partición de entrada (`select`, `filter`, `withColumn`).
- **Dependencia ancha** (*wide*): una partición de salida depende de varias
  particiones de entrada, lo que obliga a un *shuffle* (`groupBy`,
  `orderBy`, `join` sin `broadcast`, `distinct`).

## Crear una SparkSession local

Construimos una única `SparkSession` en modo `local[2]`: dos hilos que
simulan dos ejecutores dentro del mismo proceso Python/JVM del kernel. Esta
sesión es la que reutilizaremos en casi todas las secciones siguientes.

La memoria del driver se deja deliberadamente modesta. Esta JVM comparte el
contenedor `namenode` con el propio NameNode, el ResourceManager, el
Timeline Server y el kernel de Jupyter (recuerda la tabla de límites de S1:
2 vCPU y 4096 MiB para todo el contenedor). Pedir varios gigabytes de
memoria del driver aquí dejaría sin margen a los demonios de Hadoop; para
los volúmenes de SF3 que leeremos con `select`/`filter`/agregaciones no hace
falta.

También fijamos `spark.sql.files.maxPartitionBytes` a `64m`, el mismo valor que
`dfs.blocksize` en este clúster (S1). Por defecto Spark parte la lectura en
trozos de hasta 128 MiB, que aquí abarcarían dos bloques HDFS, quizá en
DataNodes distintos; con 64 MiB, cada trozo equivale a un bloque. Spark no
alinea las particiones con los bloques por sí solo: lo hace este parámetro, y
sólo si lo ajustas tú. Con las tablas de esta sesión el efecto en el rendimiento
es pequeño; lo que interesa es entender qué relaciona la lectura de Spark con
el almacenamiento que hay debajo.

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

spark: SparkSession = (
    SparkSession.builder.appName("TCDM_S4_Local")
    .master("local[2]")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)
sc: SparkContext = spark.sparkContext

print(f"Spark version: {spark.version} | Master: {sc.master} | App: {sc.appName}")

`spark.sql.shuffle.partitions` controla cuántas particiones produce Spark
tras una operación *wide* (por ejemplo, un `groupBy`). El valor por defecto
es 200, pensado para clústeres grandes; lo bajamos a 8 porque en `local[2]`,
con los volúmenes de SF3, 200 particiones diminutas sólo añadirían
coordinación sin paralelismo real.

### La interfaz web de Spark

Mientras exista una `SparkSession`, su driver sirve una interfaz web en el
puerto 4040. El driver de esta sesión vive en `namenode`, y
`compose-hadoop-cluster.yml` publica ese puerto en tu equipo: ábrela en
<http://localhost:4040> y déjala en una pestaña del navegador durante toda
la sesión. Se actualiza con cada acción que ejecutes y sigue disponible,
con todo lo ejecutado hasta entonces, hasta que se llame a `spark.stop()`.

| Pestaña | Qué muestra | Para qué sirve en esta sesión |
| --- | --- | --- |
| **Jobs** | Un *job* por cada acción (`show`, `count`, una escritura…), con su estado, su duración y una línea de tiempo | Comprobar que una transformación no lanza nada y que una acción sí |
| **Stages** | Las etapas de cada *job* y sus tareas: cuántas son, qué ejecutor hizo cada una, cuánto tardó y cuántos datos leyó o intercambió en el *shuffle*. «DAG Visualization» dibuja el grafo de dependencias entre las operaciones de la etapa | Ver dónde hay un *shuffle* (dependencia ancha) y cuántas tareas genera cada número de particiones |
| **Storage** | Los DataFrames persistidos con `cache()` o `persist()`, qué fracción está materializada y cuánto ocupa en memoria y en disco | Sección de persistencia |
| **Environment** | La configuración efectiva: propiedades de Spark, versiones de Java y Scala, rutas de bibliotecas | Confirmar valores como `spark.sql.shuffle.partitions` |
| **Executors** | Los ejecutores registrados, con sus núcleos, su memoria, las tareas completadas y el tiempo de recolección de basura. En `local[2]` sólo aparece `driver` | Sobre YARN, ver un ejecutor por contenedor y en qué DataNode está cada uno |
| **SQL / DataFrame** | Una entrada por consulta ejecutada, con el plan físico dibujado como grafo y las métricas reales de cada operador (filas producidas, ficheros y bytes leídos, registros de *shuffle*). «Details» muestra los planes lógico y físico en texto | Es la versión gráfica, y con medidas, de `explain("formatted")` |

`explain()` muestra el plan *antes* de ejecutar; la pestaña **SQL /
DataFrame** muestra ese mismo plan *después*, con lo que ocurrió de verdad
en cada operador. Las dos vistas se complementan durante toda la sesión.

Si tienes abierto otro notebook con su propia `SparkSession`, la segunda
interfaz se sirve en el puerto 4041, que no está publicado: cierra antes la
otra sesión con `spark.stop()`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Antes de ejecutar las lecturas siguientes: ¿cuántas particiones de ejecución esperas para <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>item</code> y cuántas para <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code>?</li><li style='margin:10px 0;list-style-position:outside'>¿De qué crees que depende ese número: de los ficheros, de su tamaño, de los núcleos?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

## Primera lectura de Parquet

Leemos primero una tabla pequeña, `item`, y después una tabla de
hechos, `web_sales`. HDFS es el sistema de ficheros por defecto de este
clúster (igual que en S1 y S2), así que no hace falta ningún prefijo
`hdfs://`: basta la ruta absoluta `/datalake/raw/tpcds/<tabla>`.

In [ ]:
# Qué significa cada columna de item, la dimensión de productos; printSchema dará sus tipos.
%esquema item

In [ ]:
from pyspark.sql.dataframe import DataFrame

item_path: str = "/datalake/raw/tpcds/item"
item: DataFrame = spark.read.parquet(item_path)

print(f"Ruta física leída: {item_path}")
item.printSchema()

`spark.read.parquet(...)` **no ha leído todavía ninguna fila**: sólo ha
abierto los metadatos Parquet para conocer el esquema. Es una
transformación, perezosa como todas. Las siguientes dos llamadas sí son
acciones: obligan a Spark a ejecutar el plan sobre los datos.

In [ ]:
item.show(5)

n_item: int = item.count()
print(f"Filas en item: {n_item}")
print(f"Particiones de ejecución: {item.rdd.getNumPartitions()}")

assert n_item == 36_000

Ahora repetimos la misma secuencia con `web_sales`, la tabla de hechos que
usaremos en la mayor parte de la sesión. Fíjate en que el número de
particiones de ejecución no tiene por qué coincidir con el número de
ficheros Parquet que escribió S2. Spark parte los ficheros en trozos de,
como mucho, `spark.sql.files.maxPartitionBytes` (128 MiB por defecto, 64 MiB en nuestra
sesión para igualar el tamaño de bloque HDFS);
pero cuando los ficheros son pequeños reparte además sus bytes entre los
núcleos disponibles, para que todos tengan algo que hacer (sin hacer trozos
menores de 4 MiB, `spark.sql.files.openCostInBytes`): el número
depende también del paralelismo (dos hilos en `local[2]`; en la sección de
YARN, uno por ejecutor). Y un fichero Parquet sólo se puede partir por sus
*row groups*, los bloques de filas en que se organiza por dentro: si sólo
tiene uno, uno de los trozos se lleva todas las filas y los demás quedan
vacíos. S2 escribió `web_sales` como un único fichero, pero con grupos de
122.880 filas elegidos para que una tabla de hechos tenga muchos más grupos
que ejecutores, así que Spark puede repartirlo. Tras la lectura listamos los
ficheros de `web_sales` para que lo compares, y contamos sus grupos.

In [ ]:
# Qué significa cada columna de web_sales y a qué dimensión apunta cada clave.
%esquema web_sales

In [ ]:
web_sales_path: str = "/datalake/raw/tpcds/web_sales"
web_sales: DataFrame = spark.read.parquet(web_sales_path)

print(f"Ruta física leída: {web_sales_path}")
web_sales.printSchema()

In [ ]:
web_sales.show(5)

n_web_sales: int = web_sales.count()
print(f"Filas en web_sales: {n_web_sales}")
print(f"Particiones de ejecución: {web_sales.rdd.getNumPartitions()}")

assert n_web_sales == 2_160_165

Cuenta los ficheros `.parquet` de `web_sales` y mira su tamaño frente a los
64 MiB de `maxPartitionBytes`; compara después con las particiones de
ejecución que acaba de imprimir la celda anterior.

In [ ]:
!hdfs dfs -ls -h /datalake/raw/tpcds/web_sales

Spark sólo puede partir ese fichero por sus *row groups*, así que lo que
importa no es cuántos tiene, sino que haya más que ejecutores: dos hilos
en `local[2]` y, en la sección de YARN, un ejecutor por NodeManager (tres). La celda
siguiente lee el pie del fichero, como hizo S2 en «Grupos de filas de las
tablas de hechos», sin leer ninguna columna, y comprueba que tiene más
grupos que DataNodes. Si falla, el `web_sales` de `/datalake/raw/tpcds` no
es el que escribe S2: vuelve a esa sesión.

In [ ]:
import fsspec
import pyarrow.parquet as pq
from fsspec.spec import AbstractFileSystem

# El mismo filesystem de hdfs-native que en S2 y S3.
hdfs: AbstractFileSystem = fsspec.filesystem("hdfs", host="namenode", port=9000)
# Los tres DataNodes de S1: en la sección de YARN habrá un ejecutor en cada uno.
DATANODES: int = 3

web_sales_files: list[str] = sorted(
    path
    for path in hdfs.glob(f"{web_sales_path}/*")  # type: ignore
    if not path.endswith("/_SUCCESS")
)
web_sales_groups: int = 0
for path in web_sales_files:
    with hdfs.open(path, "rb") as source:
        web_sales_groups += pq.ParquetFile(source).metadata.num_row_groups
print(f"Grupos de filas de web_sales: {web_sales_groups}")

assert web_sales_groups > DATANODES, (
    f"web_sales tiene {web_sales_groups} grupos de filas y necesita más que "
    f"DataNodes ({DATANODES}) para repartirse entre todos los ejecutores"
)

El objetivo de esta lectura no es traer SF3 completo a la memoria del
driver: es distribuir la lectura y el cómputo entre particiones que Spark
puede procesar en paralelo. `web_sales` es sólo una parte de los 545 MiB que
ocupan en Parquet las 18 tablas de SF3 (su tamaño exacto lo da
`hdfs dfs -du -h /datalake/raw/tpcds/web_sales`); TPC-DS a escalas mayores (SF100, SF1000...) seguiría funcionando
con el mismo código precisamente porque el driver nunca materializa la
tabla entera, sólo coordina el trabajo sobre sus particiones.

Mira ahora la pestaña **Jobs** de <http://localhost:4040>: deben aparecer
los *jobs* que han lanzado los `show()` y los `count()` de estas celdas, y
ninguno por las dos llamadas a `spark.read.parquet(...)`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Spark también lee los Parquet de S2</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>spark.read.parquet(...)</code> sólo abre metadatos: leer el esquema no lanza una acción</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>show()</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>count()</code> son acciones: ahí se ejecuta el plan</li><li style='margin:10px 0;list-style-position:outside'>El número de particiones de ejecución no coincide con el de ficheros: Spark lo decide por tamaño, no por directorio</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">El DataFrame de Spark no es el de pandas</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>No guarda filas: describe un cálculo sobre particiones; las filas llegan con una acción</li><li style='margin:10px 0;list-style-position:outside'>Es inmutable: cada transformación devuelve un DataFrame nuevo; no hay <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>df[&quot;x&quot;] = ...</code></li><li style='margin:10px 0;list-style-position:outside'>No tiene índice: ni <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>loc</code> ni <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>iloc</code>, ni orden de filas salvo que se pida con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>orderBy</code></li><li style='margin:10px 0;list-style-position:outside'>Las columnas son expresiones: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>col(&quot;x&quot;) &gt; 0</code> es una condición que se evalúa al ejecutar</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>select</code> ≈ <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SELECT</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>filter</code> ≈ <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>WHERE</code>, <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>withColumn</code> ≈ <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>SELECT *, expresión AS nombre</code></li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Transformaciones, acciones y planificación

Hasta aquí sólo se han leído tablas y se han lanzado acciones sobre ellas
(`show`, `count`). Para construir una consulta se encadenan
**transformaciones**: métodos de un DataFrame que devuelven otro DataFrame.
Las tres más básicas son `select`, `filter` y `withColumn`, las que
aparecían como ejemplo de dependencia estrecha en «Términos que reaparecerán
en los planes de ejecución». Antes de usarlas conviene ver en qué se
diferencia el DataFrame de Spark del de pandas, que es el que ya conoces.

### El DataFrame de Spark frente al de pandas

Los dos son tablas con columnas con nombre y tipo, y muchas operaciones
tienen un equivalente directo. Pero hay cuatro diferencias que cambian la
forma de escribir el código:

- **No guarda las filas.** Un DataFrame de pandas tiene todos sus datos en
  la memoria de tu proceso. Uno de Spark es la descripción de un cálculo
  (un plan) sobre datos repartidos en particiones, que pueden estar en
  otras máquinas. Por eso `print(web_sales)` muestra el esquema y no filas:
  las filas se piden con una acción (`show`, `count`, `collect` o
  `toPandas`, que trae el resultado al driver como un DataFrame de pandas
  y sólo tiene sentido si es pequeño).
- **Es inmutable.** No existe `df["nueva"] = ...` ni `inplace=True`: cada
  transformación devuelve un DataFrame nuevo y deja el original intacto.
  Por eso las transformaciones se encadenan con `.` y el resultado se
  guarda en otra variable.
- **No tiene índice.** No hay `loc` ni `iloc`, ni una «fila 3»: las filas
  están repartidas entre particiones y no tienen un orden garantizado salvo
  que se pida con `orderBy`. Las filas se eligen siempre por el contenido
  de sus columnas.
- **Las columnas son expresiones.** En pandas, `df["x"]` es una serie con
  todos los valores. En Spark, `col("x")` (o `web_sales["x"]`) es sólo una
  referencia a la columna, y `col("x") > 0` construye otra expresión, una
  condición, que Spark evaluará en cada fila cuando ejecute el plan.

La tabla relaciona las operaciones de esta sección con pandas y con el SQL
que se escribió con DuckDB en S2:

| Qué se quiere | pandas | Spark | SQL |
| --- | --- | --- | --- |
| Quedarse con unas columnas | `df[["a", "b"]]` | `df.select("a", "b")` | `SELECT a, b` |
| Quedarse con unas filas | `df[df["a"] > 0]` o `df.query("a > 0")` | `df.filter(col("a") > 0)` o `df.filter("a > 0")` | `WHERE a > 0` |
| Añadir o recalcular una columna | `df["m"] = df["a"] / df["b"]` | `df.withColumn("m", col("a") / col("b"))` | `SELECT *, a / b AS m` |
| Renombrar o quitar columnas | `df.rename(columns={"a": "x"})`, `df.drop(columns="a")` | `df.withColumnRenamed("a", "x")`, `df.drop("a")` | `a AS x` |
| Ver unas pocas filas | `df.head()` | `df.show(5)` | `LIMIT 5` |

### `select`, `filter` y `withColumn`

- **`select(...)`** decide qué columnas tendrá el resultado y en qué orden.
  Acepta nombres (`"ws_order_number"`), `"*"` para todas, y expresiones con
  nombre propio: `(col("ws_net_profit") * 100).alias("beneficio_centimos")`
  calcula una columna nueva y `alias` le da nombre. Es el `SELECT` de SQL;
  su variante `selectExpr("ws_net_profit * 100 AS beneficio_centimos")`
  acepta directamente texto SQL.
- **`filter(condición)`**, o su sinónimo `where`, conserva las filas en las
  que la condición es cierta. La condición puede ser una expresión de
  columna (`col("ws_net_profit") > 0`) o un texto SQL
  (`"ws_net_profit > 0"`): las dos producen el mismo plan. Varias
  condiciones se combinan con `&` (y), `|` (o) y `~` (no), cada una entre
  paréntesis, igual que las máscaras booleanas de pandas; `and` y `or` de
  Python dan error. Las columnas tienen métodos para las condiciones
  habituales: `isNotNull()`, `isin(...)` o `between(mínimo, máximo)`. Una
  fila en la que la condición es nula (por ejemplo, porque compara un valor
  nulo) se descarta, como en el `WHERE` de SQL.
- **`withColumn(nombre, expresión)`** devuelve todas las columnas del
  DataFrame más una calculada fila a fila. Si ya existe una columna con ese
  nombre, la sustituye, lo que sirve, por ejemplo, para cambiar un tipo:
  `withColumn("x", col("x").cast("double"))`. Para varias columnas a la vez
  está `withColumns({"a": ..., "b": ...})`.

Las tres trabajan fila a fila dentro de cada partición, sin necesitar filas
de otras: por eso son dependencias estrechas. Ninguna lee datos; sólo añaden
un paso al plan. Lo que sí hace Spark al momento es comprobar que las
columnas existen: un nombre mal escrito falla ya en el `select`, no al
lanzar la acción. Las funciones para construir expresiones más elaboradas
(fechas, condiciones, agregados) están en el módulo `pyspark.sql.functions`,
que se presenta en «Funciones de columna de PySpark (`F`)»; de él viene ya
`col`.

La celda siguiente encadena las tres sobre `web_sales`: se queda con cuatro
columnas, conserva las líneas con beneficio positivo y añade el margen de
cada línea sobre su precio de venta.

In [ ]:
from pyspark.sql.functions import col

web_sales_profitable: DataFrame = (
    web_sales.select(
        "ws_order_number",
        "ws_bill_customer_sk",
        "ws_net_profit",
        "ws_ext_sales_price",
    )
    .filter(col("ws_net_profit") > 0)
    .withColumn("margen_sobre_venta", col("ws_net_profit") / col("ws_ext_sales_price"))
)

print(f"Tipo del objeto: {type(web_sales_profitable)}")

Sin lanzar ninguna acción se pueden comprobar varias de las afirmaciones
anteriores: que una condición es una expresión y no un conjunto de valores,
que `withColumn` no ha modificado `web_sales`, y que `filter` con texto SQL
describe el mismo cálculo que con una expresión de columna.
`sameSemantics` compara los dos planes sin ejecutarlos.

In [ ]:
from pyspark.sql.column import Column

condicion: Column = col("ws_net_profit") > 0
print(f"Una condición es una expresión, no datos: {condicion}")
print(f"El DataFrame sólo describe su esquema: {web_sales_profitable}")

print(
    f"web_sales conserva sus {len(web_sales.columns)} columnas; "
    f"¿tiene margen_sobre_venta?: {'margen_sobre_venta' in web_sales.columns}"
)

filtro_sql: DataFrame = web_sales.filter("ws_net_profit > 0")
mismo_calculo: bool = filtro_sql.sameSemantics(web_sales.filter(condicion))
print(f'¿filter("ws_net_profit > 0") equivale a filter(col(...) > 0)?: {mismo_calculo}')

assert "margen_sobre_venta" not in web_sales.columns
assert mismo_calculo

`web_sales_profitable` es un DataFrame con un plan pendiente, no un
resultado. La siguiente celda dispara una acción (`show`) y es el momento
en el que Spark realmente lee, filtra y calcula.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Antes de ejecutar <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>show(5)</code>: ¿cuántos <em>jobs</em> nuevos esperas en la pestaña <strong>Jobs</strong>?</li><li style='margin:10px 0;list-style-position:outside'>¿Y si en lugar de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>show(5)</code> ejecutaras <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>count()</code>? ¿Qué cambia: el número de filas que se leen, o el de <em>stages</em>?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

In [ ]:
web_sales_profitable.show(5)

### Seleccionar pocas columnas frente a `select *`

Parquet es un formato columnar: leer sólo las columnas que hacen falta
reduce los bytes que hay que transportar desde el fichero. Comparamos el
plan físico de una consulta que sólo toca dos columnas con el de otra que
lee todas.

Un plan de `explain("formatted")` se lee de abajo arriba: el árbol de la
parte superior dibuja primero la última operación y, al final, el
`Scan parquet`, que es lo primero que ocurre; debajo, cada operador se
detalla con el número que lo identifica en el árbol, y el `(1)` es el
primero que se ejecuta. En el `Scan parquet` hay dos líneas que importan
hoy: `ReadSchema` (las columnas que se leen de verdad) y `PushedFilters`
(los filtros que se entregan al lector de Parquet).

In [ ]:
narrow_query: DataFrame = web_sales.select("ws_order_number", "ws_net_profit").filter(
    col("ws_net_profit") > 0
)
wide_query: DataFrame = web_sales.select("*").filter(col("ws_net_profit") > 0)

print("Plan de la consulta con columnas seleccionadas:")
narrow_query.explain("formatted")

In [ ]:
print("Plan de la consulta con select(\"*\"):")
wide_query.explain("formatted")

Compara la sección `ReadSchema` de ambos planes: en la consulta estrecha
sólo aparecen `ws_order_number` y `ws_net_profit`; en la de `select("*")`
aparece el esquema completo de `web_sales`. El filtro
`ws_net_profit > 0` también aparece empujado hacia el `PushedFilters` del
`Scan parquet` en los dos casos: eso es *pushdown*: Spark entrega al lector de Parquet qué columnas
necesita (*projection*) y qué condición deben cumplir las filas
(*predicate*), y el lector aprovecha que el formato es columnar, y que
guarda el mínimo y el máximo de cada columna por bloque de filas, para no
leer lo que no hace falta.
El efecto hay que comprobarlo en el plan, no
darlo por hecho sólo porque la función se llame `filter`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Las dos consultas devuelven las mismas filas filtradas: ¿qué cambia entonces entre el plan «estrecho» y el de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>select(&quot;*&quot;)</code>?</li><li style='margin:10px 0;list-style-position:outside'>El filtro aparece en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>PushedFilters</code>: ¿por qué hay que comprobarlo en el plan en lugar de darlo por supuesto porque la función se llame <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>filter</code>?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

## Filtrado, ordenación y agrupamiento

Contamos, para cada cliente, cuántas líneas de venta con beneficio positivo
tiene en `web_sales`, y mostramos los diez clientes con más líneas.

Antes de agrupar descartamos las líneas sin cliente. TPC-DS deja a propósito
algunas claves nulas en las tablas de hechos, para simular datos de origen
imperfectos, y `groupBy` reúne todos los nulos en un único grupo: sin el
filtro, ese grupo `NULL` encabezaría la lista con más líneas que cualquier
cliente real, como si fuera uno más. S2 hizo lo mismo en «Cliente con más
pedidos», con `WHERE ws_bill_customer_sk IS NOT NULL`.

Aquí seguimos
contando **líneas**, no pedidos: cada fila de `web_sales` es una línea de
un pedido, y un mismo pedido puede tener varias líneas.

In [ ]:
lineas_con_beneficio: DataFrame = (
    web_sales.filter((col("ws_net_profit") > 0) & col("ws_bill_customer_sk").isNotNull())
    .groupBy("ws_bill_customer_sk")
    .count()
    .orderBy(col("count").desc())
)
lineas_con_beneficio.show(10)

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Persistencia</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Spark recalcula un DataFrame desde sus fuentes cada vez que una acción lo necesita</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>cache()</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>persist()</code> guardan el resultado la primera vez que se materializa</li><li style='margin:10px 0;list-style-position:outside'>Compensa con los DataFrames que varias acciones reutilizan, como los de las secciones siguientes</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Persistencia

Una acción ejecuta su plan entero, desde la lectura de los Parquet: Spark no
guarda el resultado de una acción para la siguiente. En la pestaña
**SQL / DataFrame** puedes comprobarlo: `web_sales.count()`,
`web_sales_profitable.show(5)` y el agrupamiento por cliente empiezan cada
uno con su propio `Scan parquet`, que vuelve a leer `web_sales` de HDFS.

Las secciones siguientes reutilizan varias veces los mismos DataFrames. Los
cinco tipos de join lanzan una decena de acciones sobre las mismas cuatro
columnas de `web_sales`; el historial de pedidos y las funciones de ventana
parten todos de la misma agregación por pedido; y el resultado que se
escribe en «Guardar resultados» se cuenta y se escribe dos veces. Sin nada
más, cada una de esas acciones repetiría la lectura y todas las
transformaciones anteriores.

`cache()` marca un DataFrame para que Spark guarde sus particiones la
primera vez que una acción lo calcule. Las acciones siguientes leen esas
particiones de la memoria del ejecutor (en `local[2]`, la JVM del driver)
en lugar de recalcularlas. Igual que una transformación, `cache()` es
perezoso: no calcula nada hasta la siguiente acción, y esa primera acción
tarda algo más que sin caché, porque además de calcular el resultado
escribe la copia.

Persistir tiene un coste: la copia ocupa memoria que ya no pueden usar las
tareas para ordenar o agregar, y en este driver sólo hay 1 GiB. Compensa
cuando un DataFrame se va a usar varias veces y recalcularlo es caro; uno
que se usa una sola vez no gana nada. Por eso, al terminar con él, se libera
con `unpersist()`.

Empezamos por `web_sales_slim`, las cuatro columnas de `web_sales` que
usarán los joins: número de pedido, cliente, domicilio de facturación y
beneficio neto.

In [ ]:
web_sales_slim: DataFrame = web_sales.select(
    "ws_order_number", "ws_bill_customer_sk", "ws_bill_addr_sk", "ws_net_profit"
)
print(f"¿Está cacheado antes de cache()?: {web_sales_slim.is_cached}")

web_sales_slim.cache()
n_web_sales_slim: int = web_sales_slim.count()  # acción que materializa la caché

print(f"Filas en web_sales_slim: {n_web_sales_slim}")
print(f"¿Está cacheado después de cache()?: {web_sales_slim.is_cached}")
print(f"Nivel de almacenamiento: {web_sales_slim.storageLevel}")

La pestaña **Storage** lista ahora `web_sales_slim`, con el número de
particiones guardadas y lo que ocupan en memoria. La celda siguiente repite
el agrupamiento por cliente de «Filtrado, ordenación y agrupamiento», ahora
sobre `web_sales_slim`.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_slim</code> está cacheado y construyes <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_slim.filter(...)</code>: ¿lee otra vez los Parquet de HDFS o usa la caché?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué mirarías en el plan para saberlo?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

In [ ]:
lineas_con_beneficio_desde_cache: DataFrame = (
    web_sales_slim.filter((col("ws_net_profit") > 0) & col("ws_bill_customer_sk").isNotNull())
    .groupBy("ws_bill_customer_sk")
    .count()
    .orderBy(col("count").desc())
)
lineas_con_beneficio_desde_cache.show(10)

El resultado es el mismo que el de `lineas_con_beneficio`, pero esta vez
Spark no ha vuelto a leer `web_sales` de HDFS: las filas salen de la copia
en caché, y la celda tarda menos.

Ojo con lo que significa «cacheado»: `cache()` marca **ese** DataFrame. Uno
construido a partir de él con otra transformación no está marcado
(`is_cached` es `False`), pero su plan sigue partiendo de los datos que ya
están en caché. La celda siguiente imprime ese plan: en él aparece
`InMemoryTableScan`, la lectura de la caché, sobre `InMemoryRelation`, la
caché misma. Debajo sigue el `FileScan parquet`: es la receta con la que
Spark construyó la caché, y con la que recalcularía una partición que se
perdiera, no una lectura que se repita en cada acción. Lo que se reutiliza
es la parte del plan que ya estaba calculada.

In [ ]:
print(f"¿Está cacheado el DataFrame derivado?: {lineas_con_beneficio_desde_cache.is_cached}")
lineas_con_beneficio_desde_cache.explain()

### Niveles de almacenamiento

`cache()` es `persist()` con el nivel de almacenamiento por defecto. En un
DataFrame ese nivel es `MEMORY_AND_DISK_DESER` («Disk Memory Deserialized»
en la salida de más arriba): las particiones que no caben en memoria se
escriben en el disco local del ejecutor en lugar de recalcularse. Ese disco
es el del contenedor en el que corre el ejecutor, no HDFS: la copia
desaparece con la `SparkSession`. `persist(nivel)` permite elegir otro
nivel:

| Nivel | Qué hace con una partición que no cabe en memoria |
| --- | --- |
| `MEMORY_AND_DISK_DESER` (el de `cache()`) | La escribe en el disco local |
| `MEMORY_ONLY` | No la guarda: se recalcula desde las fuentes cada vez que se necesita |
| `DISK_ONLY` | Todas las particiones van al disco local; no ocupa memoria de almacenamiento |

In [ ]:
from pyspark import StorageLevel

print(
    "¿cache() ha usado MEMORY_AND_DISK_DESER?: "
    f"{web_sales_slim.storageLevel == StorageLevel.MEMORY_AND_DISK_DESER}"
)

`web_sales_slim` sigue en caché durante toda la sección de joins, y lo
liberamos al terminarla. A partir de aquí, cuando un DataFrame vaya a
reutilizarse varias veces, lo cachearemos de la misma forma y lo liberaremos
cuando ya no se use.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Persistencia</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Sin caché, cada acción recalcula el DataFrame desde los Parquet</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>cache()</code> es perezoso: lo guarda la primera acción, que tarda algo más</li><li style='margin:10px 0;list-style-position:outside'>Un DataFrame derivado no está cacheado, pero su plan parte de la caché (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>InMemoryTableScan</code>)</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>unpersist()</code> al terminar: la caché ocupa memoria del ejecutor</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">El API de DataFrames en acción</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Joins con las dimensiones: tipos, huérfanos y broadcast join</li><li style='margin:10px 0;list-style-position:outside'>Tipos complejos: un historial de pedidos como array de structs</li><li style='margin:10px 0;list-style-position:outside'>UDFs frente a funciones integradas: qué ve y qué no ve el optimizador</li><li style='margin:10px 0;list-style-position:outside'>Ventanas: máximo por cliente, lag/lead, medias móviles y acumulados</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Funciones de columna de PySpark (`F`)

En la siguiente celda importamos `pyspark.sql.functions as F`: `F` es un
alias para escribir las funciones de Spark sobre columnas. Estas expresiones
se combinan con `select`, `filter` o `agg`; Spark las incorpora al plan
antes de calcularlas sobre los datos. Estas son las que más usaremos en S4:

| Función | Qué hace en este notebook |
| --- | --- |
| `F.col("ca_country")` | Se refiere a una columna para compararla, filtrarla u ordenarla. Por ejemplo, `F.col("ca_country") == "España" construye una condición para cada fila. |
| `F.year("ws_sold_timestamp")`, `F.to_date(...)` | Extraen el año o la fecha de una marca de tiempo. Usamos el año para agrupar ventas y la fecha para ordenar pedidos. |
| `F.sum`, `F.avg`, `F.min`, `F.max`, `F.count` | Calculan agregados. `F.count("ws_ext_sales_price")` cuenta sólo los valores no nulos de esa columna; `F.count("*")` contaría todas las filas. |
| `F.first` | Toma un valor de cada grupo; no significa «el más antiguo» porque no garantiza un orden. |
| `F.when(...).otherwise(...)` | Construye una condición como `CASE WHEN` en SQL; aquí clasifica el beneficio de una venta. |
| `F.struct`, `F.collect_list`, `F.size`, `F.explode` | Forman una estructura por pedido, reúnen pedidos en un array, miden su longitud y vuelven a desplegar un elemento por fila. El orden de `collect_list` no está garantizado. |
| `F.lag`, `F.lead`, `F.datediff` | En la sección de ventanas, consultan el pedido anterior y el siguiente y calculan días entre fechas. |

Métodos como `.alias()`, `.isNotNull()`, `.desc()` y `.over()` se aplican
a la expresión de columna resultante; no son funciones del módulo `F`.
También distinguiremos `F.count(...)` del método `DataFrame.count()`,
que devuelve un número al ejecutar el recuento. La [referencia de funciones
de PySpark](https://spark.apache.org/docs/4.1.0/api/python/reference/pyspark.sql/functions.html)
recoge el catálogo completo.

## Joins

Cargamos `customer` y `customer_address`. Las ventas las toma cada join de
`web_sales_slim`, que ya está en caché.

In [ ]:
# Las dos dimensiones que se cargan a continuación y las claves que las unen.
%esquema customer customer_address

In [ ]:
import pyspark.sql.functions as F

customer: DataFrame = spark.read.parquet("/datalake/raw/tpcds/customer")
customer_address: DataFrame = spark.read.parquet("/datalake/raw/tpcds/customer_address")

n_customer: int = customer.count()
n_customer_address: int = customer_address.count()
print(f"Filas en customer: {n_customer}")
print(f"Filas en customer_address: {n_customer_address}")
assert n_customer == 188_000
assert n_customer_address == 94_000

# S2 sustituyó en cada dirección el municipio, la provincia, su código y el país
# por los de un municipio del IGN, con un JOIN interior. Este filtro reconoce las
# direcciones por los valores que dejó: país España, municipio y provincia no
# nulos y código de provincia de dos dígitos. Si no las reconoce todas, los datos
# no son los que escribió S2.
localized_addresses: DataFrame = customer_address.filter(
    (F.col("ca_country") == "España")
    & F.col("ca_city").isNotNull()
    & F.col("ca_county").isNotNull()
    & F.col("ca_state").rlike(r"^[0-9]{2}$")
)
assert localized_addresses.count() == n_customer_address

### Tipos de join

Probamos varios tipos de `join` entre `web_sales_slim` y `customer_address`
por `ws_bill_addr_sk = ca_address_sk`. No todos los domicilios facturados
en `web_sales` tienen por qué existir en `customer_address` con
integridad referencial estricta —Parquet no la impone, y TPC-DS la deja
opcional—, así que estos ejemplos también sirven para detectar huérfanos.

| `how=` | Qué devuelve | Para qué sirve aquí |
| --- | --- | --- |
| `inner` | Las líneas de venta que tienen domicilio, con las columnas de éste | El caso habitual |
| `left_outer` | Todas las líneas de venta; las columnas del domicilio quedan a `null` si no hay pareja | Ver qué líneas se quedarían sin domicilio |
| `right_outer` | Todos los domicilios, con o sin ventas | Ver domicilios sin ventas |
| `left_semi` | Sólo las líneas de venta que tienen pareja, **sin** columnas del domicilio | Filtrar por existencia (como `WHERE EXISTS`) |
| `left_anti` | Sólo las líneas de venta que **no** tienen pareja | Buscar huérfanos (como `WHERE NOT EXISTS`) |

Cada celda siguiente lanza dos acciones, un recuento y un `show`, así que
los cinco tipos de join ejecutan una decena de *jobs*. Todos leen
`web_sales_slim` de la caché, pero el join no está cacheado: cada acción lo
vuelve a calcular, y esa es la parte que más tarda. Cuenta con algunos
segundos por celda; mientras tanto, la pestaña **Jobs** muestra el avance de
cada *job*.

In [ ]:
from pyspark.sql.column import Column

join_expr: Column = web_sales_slim.ws_bill_addr_sk == customer_address.ca_address_sk

inner: DataFrame = web_sales_slim.join(customer_address, join_expr, "inner")
print(f"INNER JOIN: {inner.count()} filas.")
# Cada línea de venta aparece con las columnas de su domicilio al lado.
inner.select(
    "ws_order_number", "ws_bill_addr_sk", "ca_address_sk", "ca_city", "ca_county", "ca_state"
).show(5, truncate=False)

In [ ]:
left_outer: DataFrame = web_sales_slim.join(customer_address, join_expr, "left_outer")
print(f"LEFT OUTER JOIN: {left_outer.count()} filas (incluye líneas sin domicilio conocido).")
# Ordenamos con los nulos primero para ver las líneas sin pareja: las columnas
# del domicilio (ca_*) quedan a null.
left_outer.select(
    "ws_order_number", "ws_bill_addr_sk", "ca_address_sk", "ca_city", "ca_county", "ca_state"
).orderBy(F.col("ca_address_sk").asc_nulls_first()).show(5, truncate=False)

In [ ]:
right_outer: DataFrame = web_sales_slim.join(customer_address, join_expr, "right_outer")
print(f"RIGHT OUTER JOIN: {right_outer.count()} filas (incluye domicilios sin ventas).")
# Con los nulos primero, las filas sin ventas (ws_* a null) salen arriba si existen.
right_outer.select(
    "ws_order_number", "ws_bill_addr_sk", "ca_address_sk", "ca_city", "ca_county", "ca_state"
).orderBy(F.col("ws_order_number").asc_nulls_first()).show(5, truncate=False)

Las celdas de `LEFT OUTER` y `RIGHT OUTER` ordenan antes de mostrar cinco
filas, pero no ordenan los 2.160.165 registros. Cuando un `orderBy` va
seguido de un límite, y `show(5)` sólo pide las primeras filas, Spark lo
planifica como `TakeOrderedAndProject`: cada partición conserva únicamente
sus primeras filas según ese orden, y el driver sólo tiene que mezclar esas
pocas. Lo muestra el plan:

In [ ]:
left_outer.select("ws_order_number", "ca_address_sk").orderBy(
    F.col("ca_address_sk").asc_nulls_first()
).limit(5).explain()

In [ ]:
left_semi: DataFrame = web_sales_slim.join(localized_addresses, join_expr, "left_semi")
print(
    f"LEFT SEMI JOIN: {left_semi.count()} filas de web_sales cuyo domicilio existe"
    " y está localizado (municipio del IGN) en customer_address."
)
# Sólo columnas de web_sales_slim: el domicilio filtra, pero no se añade.
left_semi.show(5, truncate=False)

In [ ]:
left_anti: DataFrame = web_sales_slim.join(customer_address, join_expr, "left_anti")
print(
    f"LEFT ANTI JOIN: {left_anti.count()} filas de web_sales cuyo domicilio NO existe en customer_address."
)
# Los huérfanos: líneas de venta cuyo ws_bill_addr_sk no está en customer_address
# (o es null). Tampoco aquí hay columnas del domicilio.
left_anti.show(5, truncate=False)

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Compara <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>INNER</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>LEFT OUTER</code>: ¿qué te dice su diferencia?</li><li style='margin:10px 0;list-style-position:outside'>Si <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>LEFT ANTI</code> no da 0, ¿faltan domicilios en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>customer_address</code> o hay líneas con la clave a <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>null</code>? ¿Cómo lo distinguirías?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

Cómo leer los cinco recuentos: la diferencia entre `LEFT OUTER` e `INNER`
son las líneas de venta que se quedan sin pareja, y es exactamente lo que
cuenta `LEFT ANTI`. Que no sea cero no significa que falten domicilios en
`customer_address`: una clave `null` no es igual a ninguna otra, ni siquiera
a otro `null`, así que una línea sin `ws_bill_addr_sk` nunca tiene pareja
(TPC-DS deja a propósito algunas claves nulas, como vimos al agrupar por
cliente). La celda siguiente cuenta esas líneas: si su cifra coincide con la
de `LEFT ANTI`, ninguna clave apunta a un domicilio que no exista.

In [ ]:
null_address_keys: int = web_sales_slim.filter(col("ws_bill_addr_sk").isNull()).count()
print(f"Líneas de venta con ws_bill_addr_sk nulo: {null_address_keys}")

```python
# Producto cartesiano: cada fila de web_sales_slim con TODAS las filas de
# customer_address. NO debe ejecutarse: el resultado tendría
# aproximadamente 2.160.165 x 94.000 filas.
# cross = web_sales_slim.crossJoin(customer_address)
```

### Broadcast join

`customer_address` tiene 94.000 filas: es pequeña comparada con las 2.160.165
de `web_sales`. Podemos pedirle a Spark que la distribuya entera a cada
ejecutor con `broadcast()`, evitando el *shuffle* de la tabla grande. Spark ya lo hace
solo con las tablas que estima menores de `spark.sql.autoBroadcastJoinThreshold`
(10 MiB por defecto); `broadcast()` fuerza la decisión y la deja escrita en el
código.

In [ ]:
from pyspark.sql.functions import broadcast

sales_with_state: DataFrame = web_sales_slim.join(broadcast(customer_address), join_expr, "inner")

print("Plan físico del broadcast join:")
sales_with_state.explain("formatted")

En el plan busca `BroadcastHashJoin` y `BroadcastExchange`: la tabla
grande (`web_sales_slim`, que se lee de la caché con `InMemoryTableScan`)
no aparece envuelta en un `Exchange` de *shuffle*; sólo la pequeña se
distribuye entera. Para ver la alternativa, una de las actividades de más abajo desactiva ese
umbral y pide el plan sin broadcast.

Con el *broadcast join* termina el uso de `web_sales_slim` en los joins, así
que liberamos su caché. La sección de particiones de ejecución volverá a
construir planes a partir de él, ahora leyendo de los Parquet.

In [ ]:
web_sales_slim.unpersist()
print(f"¿Está cacheado web_sales_slim tras unpersist()?: {web_sales_slim.is_cached}")

## Subtotales con `rollup` y `cube`

Con `groupBy("sold_year", "i_category")` se obtiene una fila por cada pareja de
año y categoría. Un informe de ventas suele pedir además el subtotal de cada
año y el total general: `rollup` y `cube` añaden esas filas de subtotal en la
misma consulta, en lugar de lanzar un `groupBy` por cada nivel.

Para tener el año y la categoría en la misma fila unimos `web_sales` con
`item` (por `ws_item_sk = i_item_sk`), con el `join` que acabamos de ver,
y obtenemos el año de la venta con `F.year("ws_sold_timestamp")`: la
fecha y la hora de cada venta están en esa columna, y `year`, `month` u
`hour` extraen de ella la parte que haga falta. Después agregamos el
beneficio neto.

El `rollup`, el `cube` y la escritura de «Guardar resultados» parten de
este mismo join con `item`, así que lo cacheamos al construirlo. La
primera acción, la del `rollup`, lo calcula y lo guarda; el `cube` y la
escritura lo leen ya de la caché.

In [ ]:
web_sales_dated_categorized: DataFrame = web_sales.join(
    item, web_sales.ws_item_sk == item.i_item_sk
).select(F.year("ws_sold_timestamp").alias("sold_year"), "i_category", "ws_net_profit")
web_sales_dated_categorized.cache()
web_sales_dated_categorized.printSchema()

### Etiquetar los `null` antes de agregar

`rollup` y `cube` marcan las filas de subtotal con `null` en la columna que
resumen. Si los datos ya traen `null` (ventas sin fecha, artículos sin
categoría en `item`), esos grupos reales se confunden con los subtotales:
el resultado tendría varias filas con `null` y `null` sin forma de saber
cuál es cuál.

Por eso, antes de agrupar, sustituimos los `null` originales por un valor
de relleno con `coalesce`, que devuelve el primer valor que no es `null`.
Así, tras el `rollup` o el `cube`, cualquier `null` que quede es un
subtotal. Para `i_category` usamos la etiqueta `"<Sin categoría>"`; para el
año, `-1`, que no puede ser un año real y deja la columna como entero. Es un
valor *centinela*: sólo sirve para que esas ventas formen su propio grupo,
y hay que leerlo como «sin fecha», no como un año.

In [ ]:
web_sales_labeled: DataFrame = web_sales_dated_categorized.select(
    F.coalesce("sold_year", F.lit(-1)).alias("sold_year"),
    F.coalesce("i_category", F.lit("<Sin categoría>")).alias("i_category"),
    "ws_net_profit",
)
web_sales_labeled.printSchema()

In [ ]:
from pyspark.sql.functions import sum as spark_sum

rollup_year_category: DataFrame = (
    web_sales_labeled.rollup("sold_year", "i_category")
    .agg(spark_sum("ws_net_profit").alias("beneficio_neto"))
    .orderBy("sold_year", "i_category")
)
rollup_year_category.show(100)

En el resultado del `rollup` los `null` sólo aparecen en las filas de
subtotal: un `null` en `i_category` con `sold_year` relleno es el subtotal
de ese año en todas las categorías; un `null` en ambas columnas es el total
general. Las ventas sin fecha (`sold_year = -1`) y los artículos sin categoría
(`<Sin categoría>`) salen con su valor de relleno: son grupos de datos reales, con su
propia fila y su propio subtotal, y ya no se confunden con un subtotal.

`rollup` añade subtotales siguiendo el orden de las columnas indicadas
(primero por `sold_year`, luego por la combinación de ambas), pero no añade
el subtotal de cada categoría sumando todos los años.

`cube` sí genera **todas** las combinaciones de subtotales, incluida esa:

In [ ]:
cube_year_category: DataFrame = (
    web_sales_labeled.cube("sold_year", "i_category")
    .agg(spark_sum("ws_net_profit").alias("beneficio_neto"))
    .orderBy("sold_year", "i_category")
)
cube_year_category.show(100)

Comprueba en el resultado del `cube` que ahora también aparece una fila con
`sold_year` a `null` e `i_category` con un valor concreto: el beneficio de esa
categoría sumando todos los años, algo que `rollup` no calculaba. Como hemos
etiquetado los `null` originales, esas filas son siempre subtotales, incluida
la de `<Sin categoría>`, que suma los artículos sin categoría de todos los años.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cambia la consulta y lee el plan</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Repite los cinco <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>join</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_bill_customer_sk</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>customer</code>: ¿cuántas líneas se quedan sin cliente y por qué?</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_slim</code> ya no está en caché: repite el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>INNER JOIN</code> y compara en <strong>SQL / DataFrame</strong> su lectura y su duración con las de antes</li><li style='margin:10px 0;list-style-position:outside'>Desactiva el <em>broadcast</em> automático (<code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>autoBroadcastJoinThreshold = -1</code>), vuelve a construir el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>join</code> y compara el plan: ¿qué operador sustituye a <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>BroadcastHashJoin</code>?</li><li style='margin:10px 0;list-style-position:outside'>Haz el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>rollup</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_net_paid</code> en lugar de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>ws_net_profit</code>: ¿en qué fila coincide su total general con el del <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>cube</code>?</li><li style='margin:10px 0;list-style-position:outside'>En la pestaña <strong>Stages</strong>, localiza el <em>shuffle</em> de un <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>groupBy</code>: ¿qué dependencia ancha lo provoca?</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

## Funciones escalares y agregadas

Calculamos estadísticos simples sobre `ws_ext_sales_price`, el importe
extendido de venta antes de descuentos e impuestos.

In [ ]:
web_sales.select(
    F.avg("ws_ext_sales_price").alias("precio_medio"),
    F.max("ws_ext_sales_price").alias("precio_maximo"),
    F.min("ws_ext_sales_price").alias("precio_minimo"),
    F.count("ws_ext_sales_price").alias("lineas_con_precio"),
).show()

In [ ]:
web_sales.select("ws_ext_sales_price", "ws_net_profit").describe().show()

`describe()` añade además la desviación típica y los valores mínimo/máximo
como cadenas de texto (por eso todas las columnas del resultado aparecen
como `string`); es una forma rápida de comprobar rangos y detectar valores
sospechosos antes de construir una consulta de negocio más elaborada.

## Tipos complejos: arrays y structs

Spark puede representar columnas que contienen, a su vez, estructuras o
listas. Una columna anidada guarda en una sola fila lo que en un modelo
relacional ocuparía otra tabla y un `join`: es habitual cuando los datos
vienen de JSON o de una API, y Parquet la almacena por columnas sin perder
la estructura. Vamos a construir, para cada cliente, su historial de pedidos web
como un array de `struct`. Primero agregamos `web_sales` a nivel de
**pedido** (no de línea): un mismo `ws_order_number` puede tener varias
líneas, así que sumamos su importe neto pagado antes de construir el
historial.

Antes de agregar, conviene ver qué hay dentro de un pedido. Tomamos el
pedido número 1, el mismo que se siguió clave a clave en S2: cada fila es una
**línea**, es decir, un producto del pedido. El número de pedido y el cliente
se repiten en todas las filas; lo que cambia es el producto.

In [ ]:
order_lines: DataFrame = (
    web_sales.filter(F.col("ws_order_number") == 1)
    .join(item, F.col("ws_item_sk") == F.col("i_item_sk"))
    .select(
        "ws_order_number",
        "ws_bill_customer_sk",
        "ws_item_sk",
        "i_product_name",
        "ws_quantity",
        "ws_net_paid",
    )
    .orderBy("ws_item_sk")
)
order_lines.show(truncate=False)

In [ ]:
orders_base: DataFrame = (
    # Sin cliente no hay historial que construir: igual que en el filtrado de más
    # arriba, descartamos las líneas con ws_bill_customer_sk nulo, porque groupBy
    # las reuniría todas bajo un único "cliente" NULL.
    web_sales.filter(F.col("ws_bill_customer_sk").isNotNull())
    .groupBy("ws_bill_customer_sk", "ws_order_number")
    .agg(
        F.sum("ws_net_paid").alias("importe_pedido"),
        # Todas las líneas de un pedido comparten fecha: sirve cualquiera, y first() toma una.
        F.first("ws_sold_timestamp").alias("ws_sold_timestamp"),
    )
)
# El historial y las ventanas de más abajo parten de orders_base: lo cacheamos,
# y este recuento, la primera acción sobre él, lo calcula y lo guarda.
orders_base.cache()
print(f"Pedidos distintos agregados: {orders_base.count()}")

La celda anterior ha cacheado `orders_base`: el historial de pedidos y las
cuatro funciones de ventana parten de él, y sin caché cada una de esas
acciones repetiría el `groupBy` por pedido sobre todo `web_sales`.

Veamos `orders_base` con un pedido concreto. `importe_pedido` es la suma de
`ws_net_paid` de las líneas que acabamos de ver, y la fila conserva el cliente
y la marca de tiempo, que eran iguales en todas ellas:

Antes de agrupar hemos descartado las líneas sin cliente (`ws_bill_customer_sk`
nulo), como en «Filtrado, ordenación y agrupamiento». Aquí importa más aún:
cualquier agrupación posterior por cliente, la del historial, el recuento de
pedidos o el `explode`, trataría el `NULL` como un cliente más y, por tener
muchos pedidos juntos, encabezaría todas las listas.

In [ ]:
orders_base.filter(F.col("ws_order_number") == 1).show(truncate=False)

In [ ]:
customer_order_history: DataFrame = orders_base.groupBy("ws_bill_customer_sk").agg(
    F.collect_list(
        F.struct(
            col("ws_order_number"),
            col("importe_pedido"),
            col("ws_sold_timestamp"),
        )
    ).alias("historial_pedidos")
)
customer_order_history.printSchema()
customer_order_history.show(5, truncate=False)

Cada elemento del array `historial_pedidos` es ya un pedido agregado
(gracias al `groupBy` anterior por `ws_order_number`), no una línea suelta.
Podemos acceder al elemento de índice 0 con `getItem` (no tiene por qué
ser el pedido más antiguo), contar cuántos
pedidos tiene cada cliente con `size`, y filtrar clientes con muchos
pedidos.

In [ ]:
customer_order_history.select(
    "ws_bill_customer_sk",
    F.col("historial_pedidos").getItem(0).alias("primer_pedido_del_array"),
    F.size("historial_pedidos").alias("numero_de_pedidos"),
).orderBy(F.col("numero_de_pedidos").desc()).show(10, truncate=False)

In [ ]:
clientes_con_muchos_pedidos: DataFrame = customer_order_history.filter(
    F.size("historial_pedidos") > 5
)
print(f"Clientes con más de 5 pedidos: {clientes_con_muchos_pedidos.count()}")

`explode` hace el camino inverso: convierte cada elemento del array en una
fila independiente, recuperando una fila por pedido en vez de una fila por
cliente.

In [ ]:
exploded_orders: DataFrame = customer_order_history.select(
    "ws_bill_customer_sk", F.explode("historial_pedidos").alias("pedido")
)
exploded_orders.select(
    "ws_bill_customer_sk",
    col("pedido.ws_order_number").alias("pedido_numero"),
    col("pedido.importe_pedido").alias("importe"),
).orderBy("ws_bill_customer_sk", "pedido_numero").show(10)

## Funciones definidas por el usuario (UDFs)

Cuando una transformación no se puede expresar con las funciones
integradas de Spark, se puede registrar una función Python como UDF. Vamos
a clasificar cada línea de venta según su beneficio neto en tres etiquetas.

In [ ]:
from decimal import Decimal

from pyspark.sql.types import StringType


@F.udf(returnType=StringType())
def clasificar_beneficio(net_profit: Decimal | None) -> str:
    # Clasifica una línea de venta según su beneficio neto (ws_net_profit).
    if net_profit is None:
        return "desconocido"
    if net_profit < 0:
        return "pérdida"
    if net_profit < 50:
        return "margen_bajo"
    return "margen_alto"


web_sales_classified: DataFrame = web_sales.withColumn(
    "clasificación_beneficio", clasificar_beneficio(col("ws_net_profit"))
)
web_sales_classified.groupBy("clasificación_beneficio").count().orderBy(
    F.col("count").desc()
).show()

**Aviso de rendimiento.** Esta UDF funciona, pero Spark no puede mirar
dentro del código Python que hemos escrito: no sabe qué hace
`clasificar_beneficio`, así que no puede aplicarle las optimizaciones de
Catalyst (reordenar, podar columnas dentro de la función) ni ejecutarla en
Tungsten como haría con una expresión nativa. Además, cada fila viaja de la
JVM al proceso Python y vuelve, lo que añade serialización. La misma
clasificación con funciones integradas —`F.when(...).otherwise(...)`— se
ejecuta enteramente dentro del motor y suele ser bastante más rápida:

In [ ]:
web_sales_classified_builtin: DataFrame = web_sales.withColumn(
    "clasificación_beneficio",
    F.when(col("ws_net_profit").isNull(), "desconocido")
    .when(col("ws_net_profit") < 0, "pérdida")
    .when(col("ws_net_profit") < 50, "margen_bajo")
    .otherwise("margen_alto"),
)
web_sales_classified_builtin.groupBy("clasificación_beneficio").count().orderBy(
    F.col("count").desc()
).show()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>La UDF y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>F.when</code> dan la misma clasificación: ¿cómo comprobarías, sin cronometrar nada, que Spark las trata de forma distinta?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué parte del plan crees que no puede optimizar Catalyst cuando hay una UDF?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

In [ ]:
print("Plan con la UDF de Python:")
web_sales_classified.explain()
print("Plan con funciones integradas:")
web_sales_classified_builtin.explain()

En el primer plan aparece un operador `…EvalPython` (`BatchEvalPython` o
`ArrowEvalPython`, según la versión): es el punto en el que las filas salen
de la JVM hacia un proceso Python y vuelven. En el segundo no hay ninguno:
la clasificación es una expresión `CASE WHEN` dentro de un `Project`, que
Spark ejecuta sin salir del motor. Esa es la huella observable de que la UDF
es opaca; el cronómetro, en cambio, depende de la máquina.

Regla de la sesión: usa una UDF sólo cuando no exista una combinación de
funciones integradas que exprese la misma lógica.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Joins, tipos complejos y UDFs</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>El tipo de join decide qué filas sin pareja sobreviven; un <em>broadcast join</em> evita el <em>shuffle</em> de la tabla grande</li><li style='margin:10px 0;list-style-position:outside'>Un array de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>struct</code> guarda el historial de pedidos de cada cliente; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>explode</code> lo devuelve a filas</li><li style='margin:10px 0;list-style-position:outside'>Una UDF de Python es opaca para el optimizador: sólo cuando no exista una función integrada equivalente</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

## Funciones de ventana

Reutilizamos `orders_base` (un pedido por fila y sólo de clientes
conocidos, con su importe neto pagado y su marca de tiempo) y nos quedamos con la fecha del pedido, que
`F.to_date` obtiene de la marca de tiempo. Los pedidos sin fecha se
descartan, porque no se pueden ordenar en el tiempo. A partir de ahí
construimos cuatro ejemplos clásicos de funciones de ventana.

In [ ]:
from pyspark.sql.window import Window, WindowSpec

orders_dated: DataFrame = orders_base.filter(col("ws_sold_timestamp").isNotNull()).select(
    col("ws_bill_customer_sk").alias("cliente"),
    col("ws_order_number").alias("pedido"),
    F.to_date("ws_sold_timestamp").alias("fecha_pedido"),
    col("importe_pedido"),
)
orders_dated.orderBy("cliente", "fecha_pedido").show(10)

### Máximo por partición: el pedido mayor de cada cliente

`Window.partitionBy` no reduce filas —a diferencia de `groupBy`—: añade una
columna calculada a cada fila original, agregada dentro de su partición.

«Partición» tiene aquí otro sentido de la palabra: el grupo de filas de una
misma ventana (las de un cliente). No es una partición de ejecución: para
reunir las filas de cada cliente Spark hace un *shuffle*, y los grupos
resultantes pueden caer en cualquier partición de ejecución.

In [ ]:
from pyspark.sql.column import Column

ventana_por_cliente: WindowSpec = Window.partitionBy("cliente")
importe_maximo_cliente: Column = F.max("importe_pedido").over(ventana_por_cliente)

orders_dated.select(
    "cliente",
    "pedido",
    "importe_pedido",
    importe_maximo_cliente.alias("pedido_mayor_del_cliente"),
).withColumn(
    "es_su_pedido_mayor", col("importe_pedido") == col("pedido_mayor_del_cliente")
).orderBy(
    "cliente", "fecha_pedido"
).show(
    15
)

### `lag`/`lead`: días entre pedidos consecutivos de un mismo cliente

Aquí sí importa el orden dentro de la ventana: la particionamos por cliente
y la ordenamos por fecha de pedido. En el primer pedido de cada cliente no hay pedido anterior, así que `lag`
devuelve `null` (y `lead`, en el último).

In [ ]:
ventana_ordenada: WindowSpec = Window.partitionBy("cliente").orderBy("fecha_pedido")

fecha_pedido_anterior: Column = F.lag("fecha_pedido", 1).over(ventana_ordenada)
fecha_pedido_siguiente: Column = F.lead("fecha_pedido", 1).over(ventana_ordenada)

orders_dated.select(
    "cliente",
    "pedido",
    "fecha_pedido",
    F.datediff("fecha_pedido", fecha_pedido_anterior).alias("dias_desde_pedido_anterior"),
    F.datediff(fecha_pedido_siguiente, "fecha_pedido").alias("dias_hasta_pedido_siguiente"),
).orderBy("cliente", "fecha_pedido").show(15)

### Media móvil de los últimos 3 pedidos por cliente

`rowsBetween(-2, 0)` define un marco que incluye la fila actual y las dos
anteriores dentro de la misma partición ordenada. En los dos primeros pedidos de cada cliente la ventana tiene menos de tres
filas y la media se calcula con las que hay.

In [ ]:
ventana_movil: WindowSpec = ventana_ordenada.rowsBetween(-2, 0)
media_movil_3_pedidos: Column = F.avg("importe_pedido").over(ventana_movil)

orders_dated.select(
    "cliente",
    "pedido",
    "fecha_pedido",
    "importe_pedido",
    media_movil_3_pedidos.alias("media_movil_3_pedidos"),
).orderBy("cliente", "fecha_pedido").show(15)

### Suma acumulada del gasto por cliente

`Window.unboundedPreceding` hasta la fila actual acumula todas las filas
anteriores de la partición: es el patrón habitual para un total corriente.

In [ ]:
ventana_acumulada: WindowSpec = ventana_ordenada.rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
gasto_acumulado: Column = F.sum("importe_pedido").over(ventana_acumulada)

orders_dated.select(
    "cliente",
    "pedido",
    "fecha_pedido",
    "importe_pedido",
    gasto_acumulado.alias("gasto_acumulado_cliente"),
).orderBy("cliente", "fecha_pedido").show(15)

In [ ]:
# orders_base no vuelve a usarse: liberamos la memoria que ocupa su caché.
orders_base.unpersist()
print(f"¿Está cacheado orders_base tras unpersist()?: {orders_base.is_cached}")

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cambia el criterio de la ventana</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Añade <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>F.rank()</code> por importe dentro de cada cliente: ¿en qué filas coincide con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>es_su_pedido_mayor</code>?</li><li style='margin:10px 0;list-style-position:outside'>Localiza con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>lead</code> al cliente que más días tarda en volver a comprar</li><li style='margin:10px 0;list-style-position:outside'>Cambia <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>rowsBetween(-2, 0)</code> por <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>rowsBetween(-1, 1)</code>: ¿cuántas filas usa la ventana en los extremos de cada cliente?</li><li style='margin:10px 0;list-style-position:outside'>Calcula la suma acumulada por año en lugar de por cliente: ¿qué cambia en <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>partitionBy</code>?</li><li style='margin:10px 0;list-style-position:outside'>Pide el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>explain()</code> de una de las ventanas: ¿qué <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>Exchange</code> aparece y por qué?</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Las mismas preguntas, ahora en SQL</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>createOrReplaceTempView</code> da un nombre SQL a un DataFrame sólo mientras dura la sesión</li><li style='margin:10px 0;list-style-position:outside'>No crea un catálogo, ni un esquema, ni una tabla Hive</li><li style='margin:10px 0;list-style-position:outside'>Repetimos las tres preguntas de negocio de S2: cliente con más pedidos, provincia con más compras y distribución horaria</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Spark SQL con vistas temporales

Un DataFrame se puede registrar como vista temporal y consultarse con SQL:

```python
web_sales.createOrReplaceTempView("web_sales")
spark.sql("SELECT ... FROM web_sales")
```

La vista desaparece al terminar la sesión: no crea ningún nombre
persistente, ni un esquema de catálogo, ni una tabla Hive. Es sólo una
forma de escribir la misma consulta con SQL en lugar de con el API de
DataFrames. Reproducimos aquí, en Spark SQL, las tres preguntas de negocio
que S2 ya respondió en SQL con DuckDB sobre los mismos ficheros,
para comprobar que el motor cambia pero el significado de los datos no.

In [ ]:
web_sales.createOrReplaceTempView("web_sales")
customer.createOrReplaceTempView("customer")
customer_address.createOrReplaceTempView("customer_address")

In [ ]:
%esquema customer

### Cliente con más pedidos

Como en S2, contamos pedidos **distintos** con
`count(DISTINCT ws_order_number)`: contar `*` contaría líneas, no pedidos.

In [ ]:
spark.sql("""
    SELECT
        c.c_customer_id,
        c.c_first_name,
        c.c_last_name,
        count(DISTINCT s.ws_order_number) AS pedidos,
        count(*) AS lineas_de_venta
    FROM web_sales AS s
    JOIN customer AS c
        ON s.ws_bill_customer_sk = c.c_customer_sk
    GROUP BY c.c_customer_id, c.c_first_name, c.c_last_name
    ORDER BY pedidos DESC, c.c_customer_id
    LIMIT 10
""").show()

### Provincia con más compras

S2 cruzó las direcciones sintéticas con los municipios del IGN.
`ca_county` contiene ahora la provincia y `ca_state` su código INE de dos
caracteres; las claves que enlazan las ventas no cambiaron.

In [ ]:
spark.sql("""
    SELECT
        a.ca_state AS codigo_provincia,
        a.ca_county AS provincia,
        count(DISTINCT s.ws_order_number) AS pedidos,
        sum(s.ws_ext_sales_price) AS ventas
    FROM web_sales AS s
    JOIN customer_address AS a
        ON s.ws_bill_addr_sk = a.ca_address_sk
    GROUP BY a.ca_state, a.ca_county
    ORDER BY pedidos DESC, a.ca_state
    LIMIT 10
""").show()

### Distribución horaria de las compras

No hace falta ninguna dimensión: `hour()` extrae la hora, de 0 a 23, de
`ws_sold_timestamp`. Las ventas sin fecha se excluyen, y las que no tenían
hora quedaron a medianoche en S2, así que cuentan en la hora 0.

In [ ]:
spark.sql("""
    SELECT
        hour(ws_sold_timestamp) AS hora,
        count(DISTINCT ws_order_number) AS pedidos,
        sum(ws_ext_sales_price) AS ventas
    FROM web_sales
    WHERE ws_sold_timestamp IS NOT NULL
    GROUP BY hora
    ORDER BY hora
""").show(24)

Las tres consultas anteriores son equivalentes, columna a columna y clave a
clave, a las que S2 resolvió con DuckDB. Lo único que ha
cambiado es el motor que las ejecuta y el hecho de que ahora puede
distribuirse sobre YARN, como veremos más adelante.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Las preguntas de S2, ahora con Spark SQL</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Vistas temporales: un nombre para la sesión, sin catálogo ni tabla Hive</li><li style='margin:10px 0;list-style-position:outside'>Mismo dataset, mismas claves, mismas métricas: cambia el motor, no el significado</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>count(DISTINCT ws_order_number)</code> sigue contando pedidos; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>count(*)</code>, líneas</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

### Beneficio neto por año

Esta consulta no viene de S2: es la misma agregación que ejecutará más
adelante la sesión sobre YARN (`spark_sum(ws_net_profit)`,
`count(DISTINCT ws_order_number)` y `count(*)`, agrupando por el año de
`ws_sold_timestamp`, sin las ventas que no tienen fecha).
Guardamos el resultado en local para compararlo con el que produzca YARN
sobre los mismos datos.

In [ ]:
from pyspark.sql import Row

sales_by_year_local: DataFrame = spark.sql("""
    SELECT
        year(ws_sold_timestamp) AS sold_year,
        sum(ws_net_profit) AS beneficio_neto,
        count(DISTINCT ws_order_number) AS pedidos,
        count(*) AS lineas_de_venta
    FROM web_sales
    WHERE ws_sold_timestamp IS NOT NULL
    GROUP BY sold_year
    ORDER BY sold_year
""")
sales_by_year_local.show()

# Se guarda como filas de Python: la SparkSession local se cierra antes de
# crear la sesión sobre YARN y el DataFrame deja de poder usarse.
sales_by_year_local_rows: list[Row] = sales_by_year_local.collect()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Escribir y paralelizar con criterio</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Un fichero por partición; las vacías no llegan a escribir ninguno</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>cache()</code> evita repetir el cálculo en cada escritura</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition</code> y <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>coalesce</code>: mismo verbo, costes muy distintos</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Guardar resultados

Escribimos la agregación de beneficio por año y categoría en HDFS, bajo una
ruta de trabajo separada de `/datalake`. Es un `groupBy` normal sobre
`web_sales_dated_categorized`, que sigue en caché desde el `rollup`, no el
`rollup` de más arriba: aquí no hace falta ninguna fila de subtotal, sólo
el detalle por año y categoría.

Ese resultado se usa tres veces: lo contamos y lo escribimos de dos formas
distintas. Cada una de esas acciones ejecutaría otra vez el `groupBy`, así
que lo cacheamos y el recuento, la primera acción, lo calcula y lo guarda.
Si pides el plan de una de las escrituras, por ejemplo con
`sales_by_year_category.coalesce(4).explain()`, verás `Coalesce` aplicado
directamente sobre un `InMemoryTableScan`: el `groupBy` y el join con `item`
sólo aparecen por debajo, como receta de la caché.

Primero preparamos el espacio de trabajo de esta sesión.

In [ ]:
!hdfs dfs -mkdir -p /user/luser/s4

In [ ]:
sales_by_year_category: DataFrame = web_sales_dated_categorized.groupBy(
    "sold_year", "i_category"
).agg(spark_sum("ws_net_profit").alias("beneficio_neto"))
sales_by_year_category.cache()
print(f"Filas a escribir: {sales_by_year_category.count()}")

### El problema de los ficheros pequeños

Si repartimos el resultado en muchas particiones antes de escribir, Spark
crea hasta un fichero Parquet por partición (las particiones vacías no
llegan a escribir fichero), y la mayoría de las que sí escriben quedan
casi vacías: es el problema de los «ficheros pequeños»: cada fichero y cada bloque ocupan
memoria en el NameNode (S1) y leer un fichero exige abrirlo y lanzar una
tarea, así que leer miles de ficheros diminutos es mucho más lento que leer
unos pocos grandes, tanto para HDFS como para los motores que después lean
esa tabla.

Para listar el resultado usamos una orden de shell con una variable de
Python entre llaves: en las celdas `!orden`, IPython sustituye `{variable}`
por su valor antes de ejecutar la orden.

Cuando llegues al listado, cuenta los ficheros `part-` y mira su tamaño: el
resultado sólo tiene unas decenas de filas.

In [ ]:
many_files_path: str = "/user/luser/s4/sales-by-year-category-muchos-ficheros"
sales_by_year_category.repartition(50).write.mode("overwrite").parquet(many_files_path)

In [ ]:
!hdfs dfs -ls {many_files_path}

### Consolidar antes de escribir con `coalesce`

`coalesce(n)` reduce el número de particiones sin forzar un *shuffle*
completo (a diferencia de `repartition(n)`), agrupando particiones
existentes. Es la forma habitual de evitar el problema anterior sin pagar
el coste de una redistribución completa.

In [ ]:
few_files_path: str = "/user/luser/s4/sales-by-year-category-pocos-ficheros"
sales_by_year_category.coalesce(4).write.mode("overwrite").parquet(few_files_path)

In [ ]:
!hdfs dfs -ls {few_files_path}

En esta sección `coalesce` controla cuántos ficheros produce la escritura.
Con las dos escrituras hechas, ni el resultado ni el join del que parte
vuelven a usarse: la celda siguiente libera sus cachés.

In [ ]:
sales_by_year_category.unpersist()
web_sales_dated_categorized.unpersist()
print(f"¿Está cacheado sales_by_year_category?: {sales_by_year_category.is_cached}")
print(f"¿Está cacheado web_sales_dated_categorized?: {web_sales_dated_categorized.is_cached}")

S5 retomará el control del número de ficheros al organizar físicamente los
datos por valores de columna.

## Particiones de ejecución

`repartition(n)` y `coalesce(n)` cambian cuántas particiones tiene un
DataFrame, pero tienen costes distintos: `repartition` siempre hace un
*shuffle* completo (puede aumentar o reducir el número de particiones,
redistribuyendo todas las filas); `coalesce` sólo puede reducir particiones
y evita el *shuffle* completo agrupando particiones vecinas, aunque a
costa de un reparto potencialmente menos equilibrado.

### Reparticionar antes de un join

Si dos DataFrames que se van a unir se reparticionan por la misma clave de
unión y con el mismo número de particiones, Spark puede reutilizar ese
reparto en el join en lugar de repetirlo. Pero `repartition` es en sí un
*shuffle*: no lo evita, lo adelanta. Sólo compensa cuando el DataFrame
reparticionado se va a unir o agregar varias veces por esa clave. Y no es
automático ni garantizado: hay que comprobarlo en el plan, no darlo por
supuesto por haber llamado a `repartition`.

In [ ]:
print(f"web_sales_slim particiones (antes): {web_sales_slim.rdd.getNumPartitions()}")
print(f"customer_address particiones (antes): {customer_address.rdd.getNumPartitions()}")

web_sales_by_addr: DataFrame = web_sales_slim.repartition(8, "ws_bill_addr_sk")
customer_address_by_key: DataFrame = customer_address.repartition(8, "ca_address_sk")

print(f"web_sales_by_addr particiones (después): {web_sales_by_addr.rdd.getNumPartitions()}")
print(
    f"customer_address_by_key particiones (después): {customer_address_by_key.rdd.getNumPartitions()}"
)

In [ ]:
join_expr_repartitioned: Column = (
    web_sales_by_addr.ws_bill_addr_sk == customer_address_by_key.ca_address_sk
)
sales_with_state_repartitioned: DataFrame = web_sales_by_addr.join(
    customer_address_by_key, join_expr_repartitioned, "inner"
)

print("Plan físico tras reparticionar por la clave de unión:")
sales_with_state_repartitioned.explain("formatted")

Lee el plan con dos preguntas. ¿Hay un `Exchange hashpartitioning(...)` en
cada lado? Es el *shuffle* de los dos `repartition` de la celda anterior, no
uno del `join`. ¿Y el operador de unión es `SortMergeJoin` o
`BroadcastHashJoin`? Si es `SortMergeJoin`, el `join` ha reutilizado el
reparto en lugar de repetirlo; si es `BroadcastHashJoin`, Spark ha
considerado pequeña `customer_address` (menos de
`spark.sql.autoBroadcastJoinThreshold`) y no ha necesitado ese reparto, que
entonces no ha servido de nada. Por eso hay que medir el plan en lugar de
suponer el efecto sólo por el nombre de la función.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Rompe y mide: particiones y caché</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Escribe <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sales_by_year_category</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition(1)</code> y con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition(200)</code>: cuenta los ficheros y compara sus tamaños</li><li style='margin:10px 0;list-style-position:outside'>Compara las particiones de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales</code> con los ficheros que lista <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>hdfs dfs -ls</code>: ¿coinciden?</li><li style='margin:10px 0;list-style-position:outside'>Quita el <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>cache()</code> de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>sales_by_year_category</code>, vuelve a contar y a escribir, y compara en la pestaña <strong>Jobs</strong> la duración y las <em>stages</em> de las dos escrituras</li><li style='margin:10px 0;list-style-position:outside'>Persiste <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>web_sales_slim</code> con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>persist(StorageLevel.MEMORY_ONLY)</code> y luego con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>DISK_ONLY</code>: ¿qué cambia en la pestaña <strong>Storage</strong>?</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Particiones de ejecución</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Partición de ejecución ≠ partición Hive ≠ bloque HDFS ≠ fichero Parquet</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>repartition</code> siempre hace un <em>shuffle</em> completo; <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>coalesce</code> sólo reduce particiones, sin él</li><li style='margin:10px 0;list-style-position:outside'>Reparticionar antes de un <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>join</code> adelanta el <em>shuffle</em>: hay que comprobarlo en el plan</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

## Ejercicios con el dataset

Antes de pasar a YARN, resuelve tú mismo tres ejercicios sobre `web_sales` e
`item`, con la sesión local que sigue abierta (la celda que cierra la sesión
viene justo después de estos ejercicios). Son de dificultad fácil o media y
sólo usan lo visto hasta aquí: `join`, `groupBy`, funciones de ventana,
Spark SQL y escritura de Parquet. Escribe cada solución en la celda de
código vacía que hay debajo de su enunciado (añade más si las necesitas). Los tres ejercicios hechos son lo que tendrás que
enseñar al profesor en la reunión de evaluación de esta sesión (ver
«Evidencias para la siguiente sesión»), así que conserva las celdas con su
salida.

### Categorías con más beneficio

Con `web_sales` e `item`, calcula para cada categoría de producto
(`i_category`) el número de **pedidos distintos**, el número de **líneas de
venta** y el **beneficio neto total** (`ws_net_profit`). Ordena de mayor a
menor beneficio y quédate con las cinco primeras categorías.

- Resuélvelo con la API de DataFrames.
- Los pedidos y las líneas de venta no coinciden: explica por qué.
- Si una de las filas tiene `i_category` nula, son productos sin categoría:
  déjala en el resultado y di cuál es su papel.
- Pide `.explain("formatted")` y localiza el operador de `join` que usa
  Spark y las columnas de `web_sales` que se leen (`ReadSchema`).

Enseña: la tabla de cinco filas, el plan con el `join` y las columnas leídas
señalados, y la explicación de la diferencia entre pedidos y líneas.

### Los tres productos con más beneficio de cada año

Calcula, para cada año de venta (`year` de `ws_sold_timestamp`), los **tres
productos con mayor beneficio neto** y su posición (1, 2 o 3) dentro de ese
año. Ignora las líneas de venta sin fecha.

- Un producto se identifica por su clave de negocio `i_item_id`, no por
  `i_item_sk`: como viste en S2, un mismo producto puede tener varias
  versiones, cada una con su `i_item_sk`. Agrega por `i_item_id`.
- Primero suma el beneficio por año y producto con `groupBy`; después numera
  los productos de cada año con una función de ventana
  (`Window.partitionBy(...)` y `F.row_number()`) y quédate con los tres
  primeros.
- Ordena la ventana por beneficio descendente y, para desempatar, por
  `i_item_id`, de modo que el resultado no cambie de una ejecución a otra.

Enseña: el resultado (tres filas por año), la ventana que has definido y por
qué se agrega antes de aplicarla.

### La misma consulta en SQL, comparada y guardada

Repite en **Spark SQL** la consulta de categorías con más beneficio, con
`web_sales` e `item` como vistas temporales (`item` todavía no es una
vista: regístrala tú, como se hizo con `web_sales` en la sección de Spark
SQL).

- Comprueba que el resultado de SQL es idéntico al del DataFrame del primer
  ejercicio con `exceptAll` en los dos sentidos: ambos deben devolver 0
  filas.
- Guarda el resultado en Parquet en `/user/luser/s4/ejercicios/top-categorias`
  con `coalesce(1)` y `mode("overwrite")`, y lista con `hdfs dfs -ls` los
  ficheros que ha creado: ¿cuántos hay y cuál es su tamaño?
- Vuelve a leer la ruta y comprueba que tiene las mismas filas que el
  resultado que guardaste. Recuerda que esa salida es una carpeta de
  ficheros, no una tabla registrada.

Enseña: las dos comprobaciones con `exceptAll`, el listado de ficheros y la
lectura de vuelta.

In [ ]:
spark.stop()

Cerramos aquí la sesión local. Una `SparkSession` no puede cambiar de `master`
una vez construida, así que la sección siguiente crea una nueva, ya sobre
YARN. El resto del programa —leer Parquet, `groupBy`, `explain`...— no
cambia: lo único que cambia es dónde viven los ejecutores.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #174f7a;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#174f7a" size="8" noshade style="display:none"><div style="color:#174f7a;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#174f7a" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">De <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>local[2]</code> a los DataNodes</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Mismo programa, otro <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>master</code>: <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.master(&quot;yarn&quot;)</code> en vez de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.master(&quot;local[2]&quot;)</code></li><li style='margin:10px 0;list-style-position:outside'>El driver sigue siendo este kernel; los ejecutores pasan a ser contenedores YARN en los DataNodes</li><li style='margin:10px 0;list-style-position:outside'><code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>spark-submit</code> es la otra forma de llegar a YARN: queda para el apéndice final</li></ul><hr color="#174f7a" size="2" noshade style="display:none"></div>

## Driver, ejecutores y YARN

Hasta ahora, «driver» y «ejecutores» eran hilos dentro del mismo proceso
`local[2]`. Sobre YARN son procesos reales, potencialmente en máquinas
distintas:

- el **driver** sigue siendo el proceso que construye el plan y coordina
  la aplicación; en modo `client` (el que usaremos) vive en `namenode`,
  dentro del propio kernel (en modo `cluster` viviría en un contenedor YARN y
  lo que imprime no volvería a este notebook);
- el **ApplicationMaster** es el interlocutor de la aplicación ante el
  ResourceManager: pide y libera contenedores;
- los **ejecutores** son contenedores YARN que se ejecutan en los
  NodeManagers de los DataNodes —los mismos `datanode1`, `datanode2` y
  `datanode3` de S1— y sobre los que Spark reparte las tareas.

<p align="center" style="text-align: center;"><img src="https://dsevilla.github.io/tcdm-public/figs/anim-spark-ejecutores.gif" alt="Animación: spark-submit --master yarn se lanza desde namenode y los ejecutores calculan en datanode1, datanode2 y datanode3" width="832" style="display: block; margin: 0 auto; max-width: 100%; height: auto;"></p>

Una `SparkSession` no puede cambiar de `master` una vez construida: por eso la sesión local se
cerró con `spark.stop()` y esta sección construye otra con `.master("yarn")`.
Es la forma habitual de trabajar con Spark en un clúster desde un notebook: el
driver es el propio kernel y YARN reparte los ejecutores. Existe otra, que es
empaquetar el programa en un script y enviarlo con `spark-submit`; se usa para
trabajos sin notebook de por medio y se deja en el apéndice del final, para
mirarlo después.

### Qué Python usan el driver y los ejecutores

El driver es el kernel de este notebook, que ya usa `/opt/tcdm/venv`,
instalado en la imagen de `namenode` y comprobado con la celda `%pip` de
instalación al principio de la sesión. Es el único proceso que necesita el
paquete `pyspark` instalado para construir la `SparkSession` y lanzar la
aplicación.

Los **ejecutores** son otra historia: se ejecutan en los DataNodes, cuyas
imágenes sólo traen el `python3` de base (sin PySpark ni ningún paquete
instalado por pip). Spark distribuye a través de YARN los ficheros de soporte
de PySpark (`pyspark.zip` y `py4j.zip`) y los `.jar` de Spark, así que a cada
DataNode le basta con ese intérprete base para ejecutar el código ya
distribuido. Por eso la sesión pide `spark.pyspark.python=python3` a secas: el
`python3` del sistema en cada DataNode, que es la misma versión de Python que
la de `/opt/tcdm/venv` porque ambas imágenes parten de la misma base.

Una consecuencia práctica: el código Python que viaja a los ejecutores (la
función del informe de hosts de esta sección, o una UDF) sólo puede usar la
biblioteca estándar, porque en los DataNodes no hay más paquetes.


### Cuántos ejecutores pedir

El número de ejecutores que se solicita debe derivarse de los NodeManagers
que YARN anuncia como
`RUNNING`, no de un número fijo elegido a mano. La variable de entorno
`TCDM_SPARK_EXECUTORS`, si está definida, permite forzar otro valor, por
ejemplo para probar con menos ejecutores. La lee el kernel, que se ejecuta en
`namenode`, así que se fija en una celda del notebook con `%env`, como en S3
(`%env TCDM_SPARK_EXECUTORS=1`), y después se repiten esta celda y la que crea
la sesión; exportarla en una terminal de tu equipo no tiene efecto. Si la sesión
ya está creada hay que cerrarla antes con `spark.stop()`, porque el número de
ejecutores se fija al crearla.

Por defecto se pide **un ejecutor de un núcleo por NodeManager**: tres en total.
Es una elección didáctica, no un límite del clúster: con un ejecutor por nodo
el informe de la sección muestra a simple vista que el trabajo se reparte entre
los DataNodes. Cada NodeManager anuncia 2 vcores, así que cabrían más, y el
recurso que limita es la memoria: cada ejecutor ocupa 768 MiB de los 2560 MiB
del nodo. Tampoco se fija un ejecutor por nodo: `spark.executor.instances=3` sólo
pide tres contenedores y YARN decide dónde colocarlos.

Para un cálculo de verdad, en cambio, conviene pedir más ejecutores. Con 3
nodos de 2 núcleos hay 6 núcleos en total y el ApplicationMaster usa uno,
así que lo razonable son **5 ejecutores de un núcleo**
(`%env TCDM_SPARK_EXECUTORS=5`): ocupan los núcleos disponibles sin repartir
más de lo que hay y caben de sobra en memoria. La contrapartida es que el
informe ya no tendrá una partición por ejecutor y dos ejecutores podrán
compartir DataNode.

In [ ]:
import subprocess

result: subprocess.CompletedProcess[str] = subprocess.run(
    [
        "bash",
        "-lc",
        "yarn node -list -all 2>/dev/null | awk '$2 == \"RUNNING\" {count++} END {print count + 0}'",
    ],
    capture_output=True,
    text=True,
    check=True,
)
running_nodemanagers: int = int(result.stdout.strip() or 0)
print(f"NodeManagers en estado RUNNING: {running_nodemanagers}")

assert running_nodemanagers >= 2, (
    "Se esperan al menos dos NodeManagers RUNNING. Comprueba los DataNodes con "
    "`make -C entorno status` en una terminal de tu equipo o en la interfaz del "
    "ResourceManager (http://localhost:8088); si acabas de arrancar el clúster, "
    "espera unos segundos y repite esta celda."
)

In [ ]:
import os

requested_executors: int = int(os.environ.get("TCDM_SPARK_EXECUTORS", running_nodemanagers))
print(f"Solicitando {requested_executors} ejecutores Spark a YARN.")

### Crear la sesión sobre YARN

La sesión sobre YARN se construye igual que la local: cambian el `master` y las
opciones de recursos, ajustadas a este clúster.

- `spark.executor.instances` es el número de ejecutores de la celda anterior;
  `spark.executor.cores=1` y `spark.executor.memory=512m`, junto con
  `spark.executor.memoryOverhead=256m`, suman 768 MiB por ejecutor, dentro de
  los 2560 MiB que cada NodeManager anuncia a YARN (recuerda: de los 3072 MiB
  del contenedor DataNode, 512 MiB quedan reservados para el propio DataNode,
  el NodeManager y otros procesos auxiliares).
- `spark.yarn.am.cores` y `spark.yarn.am.memory`/`memoryOverhead` son la
  petición de recursos del ApplicationMaster, un contenedor YARN aparte de los
  ejecutores.
- `spark.scheduler.minRegisteredResourcesRatio=1.0` junto con
  `maxRegisteredResourcesWaitingTime=60s` hacen que Spark espere a tener todos
  los ejecutores solicitados (o hasta 60 segundos) antes de empezar a repartir
  tareas, evitando medir la distribución de trabajo con sólo parte de los
  ejecutores ya registrados.
- `spark.pyspark.python=python3` es el intérprete de los ejecutores (el
  `python3` de base de cada DataNode); el del driver es el del kernel.
- `spark.yarn.isPython=true` avisa a YARN de que la aplicación es de Python y
  debe llevar `pyspark.zip` a los ejecutores. Lo marca `spark-submit` al
  arrancar, pero aquí la JVM de PySpark se lanzó con la sesión local, así que
  se pide explícitamente: sin él, los ejecutores no encuentran el módulo
  `pyspark` en cuanto una tarea ejecuta código Python.
- Spark sabe dónde está el ResourceManager por `HADOOP_CONF_DIR`
  (`/opt/bd/hadoop/etc/hadoop`), que el kernel hereda del entorno de `luser`.

Crear la sesión tarda más que en local —del orden de diez a treinta
segundos—: YARN tiene que lanzar el ApplicationMaster y los contenedores de los
ejecutores, y Spark sube sus bibliotecas a HDFS. Si mientras esperas la
aplicación aparece como `ACCEPTED` en el ResourceManager, YARN la ha recibido
pero todavía espera recursos; consulta `http://localhost:8088` y el estado de
los NodeManagers antes de volver a intentarlo.

In [ ]:
import os

from pyspark.sql.session import SparkSession

assert os.environ.get("HADOOP_CONF_DIR"), (
    "El kernel no hereda HADOOP_CONF_DIR: Jupyter debe arrancarse como luser "
    "dentro de namenode (`su - luser`), como se explica en «Antes de empezar»."
)

spark: SparkSession = (
    SparkSession.builder.appName("TCDM_S4_YARN")
    .master("yarn")
    .config("spark.executor.instances", str(requested_executors))
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.memoryOverhead", "256m")
    .config("spark.yarn.am.cores", "1")
    .config("spark.yarn.am.memory", "256m")
    .config("spark.yarn.am.memoryOverhead", "256m")
    .config("spark.scheduler.minRegisteredResourcesRatio", "1.0")
    .config("spark.scheduler.maxRegisteredResourcesWaitingTime", "60s")
    .config("spark.pyspark.python", "python3")
    .config("spark.yarn.isPython", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)
sc: SparkContext = spark.sparkContext

print(f"Spark version: {spark.version} | Master: {sc.master} | Aplicación: {sc.applicationId}")
print(f"Interfaz del driver: {sc.uiWebUrl}")

In [ ]:
!yarn application -list -appStates RUNNING
!yarn node -list

Mientras la sesión esté abierta, la aplicación `TCDM_S4_YARN` aparece en el
ResourceManager como `RUNNING` (`http://localhost:8088`): es el ApplicationMaster
más los ejecutores. `yarn node -list` muestra cuántos contenedores corren en
cada NodeManager. La interfaz del driver, que `local[2]` ya servía, vuelve a
estar en <http://localhost:4040> y su pestaña **Executors** lista un ejecutor
por contenedor YARN, con el DataNode donde ha caído cada uno (YARN
decide la colocación, y puede poner dos ejecutores en un mismo nodo). Sobre YARN esa interfaz también
se consulta a través del ResourceManager: el enlace *ApplicationMaster* de la
aplicación; si el navegador acaba en una dirección con el nombre
`resourcemanager`, sustitúyelo por `localhost`.

### El mismo programa, ahora sobre YARN

Repetimos la consulta de «Beneficio neto por año» con la sesión nueva: el
driver sigue en este kernel, pero las tareas se ejecutan en los ejecutores de
los DataNodes. Como la sesión es otra, hay que volver a leer `web_sales` y a
registrar la vista temporal.

In [ ]:
web_sales_yarn: DataFrame = spark.read.parquet(web_sales_path)
web_sales_yarn.createOrReplaceTempView("web_sales")

sales_by_year_yarn: DataFrame = spark.sql("""
    SELECT
        year(ws_sold_timestamp) AS sold_year,
        sum(ws_net_profit) AS beneficio_neto,
        count(DISTINCT ws_order_number) AS pedidos,
        count(*) AS lineas_de_venta
    FROM web_sales
    WHERE ws_sold_timestamp IS NOT NULL
    GROUP BY sold_year
    ORDER BY sold_year
""")
sales_by_year_yarn.show()

sales_by_year_yarn_rows: list[Row] = sales_by_year_yarn.collect()

Ese resultado debe ser el mismo que calculó la sesión local en la sección
de Spark SQL. Lo comprobamos fila a fila, con las filas que guardamos
entonces en `sales_by_year_local_rows`. Esa variable sólo existe si «Beneficio neto
por año» se ejecutó en este mismo kernel: si lo has reiniciado, la celda siguiente
falla con `NameError` y hay que volver a ejecutar el notebook hasta esa sección.

In [ ]:
assert (
    sales_by_year_yarn_rows == sales_by_year_local_rows
), "El resultado sobre YARN no coincide con el local"
print(f"Local y YARN coinciden en los {len(sales_by_year_local_rows)} años.")

### Qué host ha procesado cada partición

Obtener el mismo resultado no demuestra dónde se hizo el trabajo. Para verlo,
`mapPartitionsWithIndex` ejecuta una función en cada partición y permite
registrar allí `socket.gethostname()`; es el único punto de la sesión en que se
baja a la API de RDD, como se anunció en «API estructurada frente a RDDs». El
informe tiene una fila por partición de `web_sales`: su número, el host del
ejecutor que la procesó y cuántas filas tenía.

In [ ]:
import socket
from collections.abc import Iterator


def describe_partition(partition_id: int, rows: Iterator[Row]) -> Iterator[tuple[int, str, int]]:
    yield (partition_id, socket.gethostname(), sum(1 for _ in rows))


partition_report: list[tuple[int, str, int]] = (
    web_sales_yarn.select("ws_order_number")
    .rdd.mapPartitionsWithIndex(describe_partition)
    .collect()
)
spark.createDataFrame(
    partition_report, schema="partition INT, executor_host STRING, records LONG"
).orderBy("partition").show()

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #0c304d;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#0c304d" size="8" noshade style="display:none"><div style="color:#0c304d;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#0c304d" style="color:inherit"><b>Pregunta guía</b></font></div><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Tres hosts aparecen en el informe: ¿significa eso que el cálculo se ha repartido entre los tres DataNodes?</li><li style='margin:10px 0;list-style-position:outside'>¿Qué columna del informe lo confirma o lo desmiente?</li></ul><hr color="#0c304d" size="2" noshade style="display:none"></div>

Cada fila del informe es una partición de `web_sales`: su número, el host
del ejecutor que la procesó y cuántas filas tenía. Mira la columna
`records` antes de mirar los hosts: una tarea que procesa una partición con
0 filas ha recibido trabajo, pero no lo ha hecho. Con tres ejecutores de un
núcleo, Spark parte el fichero de `web_sales` en tres trozos, y como el
fichero tiene más *row groups* que DataNodes (lo comprobamos al leerlo en
«Primera lectura de Parquet»), los tres reciben filas: unos cientos de miles
cada uno. Si el fichero tuviera un único *row group* no se podría partir: una
partición se llevaría todas las filas y las demás quedarían vacías, porque
Spark reparte tareas, no filas. Que haya varios hosts en el informe prueba
que YARN repartió *tareas* entre DataNodes; que el *cálculo* se repartiera
lo dice la columna `records`.

In [ ]:
!yarn application -list -appStates RUNNING

Por último, extraemos del informe de particiones la lista de hosts
distintos que han recibido alguna partición y comprobamos que al menos dos
DataNodes remotos participan cuando los recursos lo permiten. Si sólo
aparece uno, repite la celda del informe: con tan pocos datos el planificador puede
concentrar el trabajo en un único nodo.

In [ ]:
executor_hosts: set[str] = {host for _, host, _ in partition_report}

print(f"Hosts que procesaron al menos una partición: {sorted(executor_hosts)}")
print(f"Ejecutores remotos que trabajaron: {len(executor_hosts)}")
assert len(executor_hosts) >= 2, "Se esperan al menos dos hosts remotos distintos"

Si todo ha ido bien, el resultado de negocio es el mismo que ya vimos en
modo local en la sección de Spark SQL —beneficio neto, pedidos distintos y
líneas de venta por año, comparados más arriba con un `assert`—, sólo que
esta vez
con las tareas repartidas entre ejecutores de, al menos, dos DataNodes
distintos y coordinadas por un driver que vive en `namenode`.

### Cerrar la sesión YARN

Hay que cerrar la sesión cuando se termina: mientras siga abierta mantiene
reservados el ApplicationMaster y los ejecutores (768 MiB cada uno), y otra
aplicación —por ejemplo, el apéndice de `spark-submit`— se quedaría en
`ACCEPTED` esperando recursos. Al llamar a `spark.stop()` la aplicación termina
y YARN libera los contenedores; después, `yarn application -list -appStates ALL`
la muestra como `FINISHED`.

In [ ]:
spark.stop()

In [ ]:
!yarn application -list -appStates ALL

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #2f6f4f;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#2f6f4f" size="8" noshade style="display:none"><div style="color:#2f6f4f;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#2f6f4f" style="color:inherit"><b>Actividades</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Cambia el reparto y vuelve a medirlo</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Cierra la sesión con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>spark.stop()</code>, ejecuta <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>%env TCDM_SPARK_EXECUTORS=1</code> y luego <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>=5</code>, vuelve a crearla en cada caso y compara hosts y duración en el ResourceManager</li><li style='margin:10px 0;list-style-position:outside'>Añade <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.repartition(12)</code> antes de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.rdd</code> en el informe de hosts: ¿cuántas particiones procesa ahora cada host? Mira la columna <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>records</code></li><li style='margin:10px 0;list-style-position:outside'>Pide <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>spark.executor.memory</code> de <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>4g</code>: ¿qué error da Spark al crear la sesión y qué dato del clúster lo explica?</li><li style='margin:10px 0;list-style-position:outside'>Abre la pestaña <strong>Executors</strong> con la sesión abierta y localiza en qué DataNode está cada ejecutor</li></ul><hr color="#2f6f4f" size="2" noshade style="display:none"></div>

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">El trabajo ocurrió en los DataNodes</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Mismo programa, otro <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>master</code>: con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>.master(&quot;yarn&quot;)</code> los ejecutores son contenedores en los DataNodes y el driver sigue en este kernel</li><li style='margin:10px 0;list-style-position:outside'>El informe registra qué host procesó cada partición y cuántas filas tenía: al menos dos DataNodes remotos recibieron tareas</li><li style='margin:10px 0;list-style-position:outside'>El número de ejecutores se derivó de los NodeManagers <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>RUNNING</code>, no de un número elegido a mano</li><li style='margin:10px 0;list-style-position:outside'>El resultado de negocio es el mismo que en local: el motor cambia, la métrica no</li></ul><hr color="#a54467" size="2" noshade style="display:none"></div>

## Hilo de negocio con TPC-DS

A lo largo de la sesión se ha mantenido la misma disciplina que en S2,
sólo que ahora ejecutada por Spark, en local y sobre YARN:

- contar líneas de `web_sales` con `count(*)` es distinto de contar
  pedidos con `count(DISTINCT ws_order_number)`, y cada consulta de esta
  sesión ha dicho explícitamente cuál de las dos calculaba;
- las sumas de importes o unidades siempre han indicado qué columna
  representa la medida (`ws_net_profit`, `ws_ext_sales_price`,
  `ss_ext_sales_price`, `ws_net_paid`...), en lugar de sumar "lo que
  hubiera";
- el año y la hora de una venta han salido siempre de `ws_sold_timestamp`,
  con `year` y `hour`, y `web_sales` se ha unido con `item`, `customer` o
  `customer_address` sólo cuando la pregunta lo necesitaba;
- ningún identificador surrogate (`ws_bill_customer_sk`, `c_customer_sk`,
  `ca_address_sk`...) se ha presentado como si fuera el nombre de una
  persona o una región administrativa real: son claves sintéticas del
  generador TPC-DS.

La sección de YARN calculó el beneficio neto por año exactamente con la
misma métrica y las mismas claves que la sección de Spark SQL calculó en
local: el motor de ejecución cambia, pero la respuesta de negocio debería
ser la misma consultando el mismo dataset inmutable.

## Preguntas para interpretar la experiencia

- ¿Qué diferencia concreta hay entre una transformación y una acción? Da un
  ejemplo de esta sesión de cada una.
- ¿Por qué `select("*")` y una selección de dos columnas producen el mismo
  resultado filtrado pero planes físicos distintos? ¿Qué cambia realmente
  entre ambos?
- El número de particiones de ejecución de un DataFrame, ¿depende del
  número de ficheros Parquet, del número de bloques HDFS o del número de
  ejecutores disponibles? Justifica la respuesta con algo que hayas medido
  en el notebook.
- ¿Por qué un `join` normal necesita un *shuffle* y un *broadcast join*
  puede evitarlo? ¿Qué condición debe cumplir la tabla que se distribuye
  entera?
- ¿Qué controla `spark.sql.shuffle.partitions` y en qué se diferencia de
  las particiones de lectura que vimos con `item` y `web_sales`?
- ¿Por qué hubo que cerrar la `SparkSession` local para trabajar sobre YARN?
  ¿Qué cambió en el código de la consulta y qué cambió en el sitio donde se
  ejecutan las tareas?
- En la sección de UDFs, ¿qué información concreta pierde Spark al no
  poder «mirar dentro» de una función Python, y qué consecuencia tiene
  eso sobre el rendimiento?
- En el informe de ejecutores, ¿qué significaría que un host apareciera con
  0 filas? ¿Qué propiedad del fichero de `web_sales` evita que ocurra?

## Evidencias para la siguiente sesión

Antes de la siguiente sesión tendrás una reunión individual breve con el
profesor para revisar el trabajo de esta sesión. Esa reunión combina una
demostración en vivo sobre tu propio ordenador y una memoria escrita breve.

### Qué mostrar en el ordenador durante la reunión

Ten preparadas estas evidencias. La primera es obligatoria: sin los tres
ejercicios hechos no se da por superada la revisión.

1. Los tres ejercicios de la sección «Ejercicios con el dataset» resueltos y
   ejecutados en tu notebook, con su salida visible: las cinco categorías
   con más beneficio (pedidos, líneas y plan), los tres productos con más
   beneficio de cada año con su ventana, y la consulta SQL comparada con
   `exceptAll` y guardada en HDFS. Tienes que saber explicar cada solución.
2. El esquema (`printSchema`) y el número de filas de `item` o
   `web_sales`, junto con su número de particiones de ejecución.
3. El plan de `.explain("formatted")` de al menos una consulta con `join`
   (por ejemplo, el *broadcast join* con `customer_address`).
4. El número de particiones de un DataFrame antes y después de un
   `repartition` o un `coalesce`.
5. La aplicación YARN completada (`yarn application -list -appStates ALL`)
   y el informe de ejecutores: al menos dos hosts distintos con alguna
   partición y las filas (`records`) de cada una, para saber si el cálculo se
   repartió de verdad.
6. El resultado de negocio (beneficio neto por año de `web_sales`) obtenido
   en local con Spark SQL y el mismo resultado obtenido sobre YARN,
   comprobando que coinciden.
7. Que sabes explicar, con tus propias palabras, la diferencia entre una
   partición de ejecución de Spark, una partición Hive y un bloque HDFS.

### Memoria escrita (una o dos páginas)

Trae también un documento breve —una o dos páginas, no hace falta más— que
no se limite a pegar capturas de las salidas anteriores: debe explicar con
tus propias palabras qué papel tienen el driver, los ejecutores y el gestor
de recursos, y qué cambia entre `local[2]` y YARN; la diferencia entre una
transformación y una acción, y qué se lee en un plan de `explain`; qué es
una partición de ejecución y en qué se distingue de un fichero Parquet, de
un bloque HDFS y de una partición Hive; y por qué el resultado de negocio
es el mismo en local y sobre YARN.

### Qué es importante de cara al examen final

El examen no pide recordar la sintaxis exacta de una función. Debes poder
explicar:

- qué hacen el driver, los ejecutores y el gestor de recursos, y qué son
  un *job*, una *stage* y una *task*;
- la evaluación perezosa: qué es una transformación, qué es una acción y
  por qué Spark construye un plan antes de ejecutar;
- la diferencia entre dependencias estrechas y anchas, qué es un *shuffle*
  y qué operaciones lo provocan;
- qué se comprueba en un plan: selección de columnas, filtros empujados a
  la lectura y tipo de `join` (incluido el *broadcast join*);
- qué controlan `repartition` y `coalesce`, y por qué importan los ficheros pequeños;
- cuándo compensa persistir un DataFrame y por qué una UDF de Python es
  más lenta que una función integrada;
- la diferencia entre contar líneas de venta y contar pedidos distintos.

<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#eaf2f8;color:#1f2933;border:1px solid #c9d6e1;border-top:10px solid #a54467;border-radius:10px;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:26px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a54467" size="8" noshade style="display:none"><div style="color:#a54467;font-size:18px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a54467" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#0c304d;font-family:Atkinson Hyperlegible,Inter,Aptos,Segoe UI,Helvetica,Arial,sans-serif;font-size:41px;line-height:1.15;margin:0 0 22px">Sesión 4</h1><ul style='margin:18px 0 8px;padding-left:30px'><li style='margin:10px 0;list-style-position:outside'>Spark lee y escribe los mismos Parquet; el driver nunca materializa la tabla entera</li><li style='margin:10px 0;list-style-position:outside'>Transformaciones perezosas, acciones que ejecutan, planes que se leen con <code style='background:#eaf2f8;color:#0c304d;padding:2px 6px;border-radius:4px;font-size:0.9em'>explain</code></li><li style='margin:10px 0;list-style-position:outside'>Local y YARN son el mismo programa: cambian el gestor de recursos y dónde viven los ejecutores</li></ul><blockquote style='background:#ffffff;color:inherit;border:none;border-left:6px solid #a54467;margin:22px 0 0;padding:12px 16px'><div style='margin-top:0px'>Siguiente bloque: buenas prácticas de data lake y particionado físico con el mismo motor (S5)</div></blockquote><hr color="#a54467" size="2" noshade style="display:none"></div>

## Detener el laboratorio

Cuando hayas guardado el notebook y terminado de revisar las evidencias,
ejecuta `make -C entorno stop` en una terminal de tu equipo, desde la raíz
de la distribución. Se detienen los servicios Compose incluidos en
`entorno/` (también RustFS) y Jupyter, pero se conservan los volúmenes
HDFS y los resultados; para continuar, ejecuta `make -C entorno hadoop-up`
y vuelve a iniciar Jupyter como en «Antes de empezar».


## Siguiente paso

S4 enseña Spark como motor. S5 utilizará ese motor para crear una tabla
refinada como colección de Parquet particionados. Todavía se conservará
una zona fuente inmutable y no se presentará la salida como una tabla
gestionada.

## Apéndice: enviar un script con `spark-submit`

Este apéndice es opcional: no se pide en la reunión de evaluación, y está para
que lo mires después. Es la otra forma de llevar un programa a YARN, y la que se
usa cuando no hay un notebook de por medio —un trabajo programado, un proceso
por lotes—. En lugar de crear la sesión desde el kernel, el programa se escribe
en un fichero `.py` que crea la suya, y `spark-submit` lo lanza pasándole los
recursos por la línea de órdenes. El cálculo es el mismo de la sección anterior;
lo que cambia es quién contiene al driver: ahora es el proceso `spark-submit`, y
el notebook sólo espera a que termine y lee lo que el trabajo ha escrito en HDFS.

Antes de ejecutar estas celdas, comprueba que la sesión de la sección anterior
está cerrada (`spark.stop()`): si no, sus contenedores siguen reservando los
recursos que este envío necesita.

### El script que se enviará a YARN

Escribimos un fichero `.py` independiente. Hace lo mismo que la sección anterior: calcula el
beneficio neto total de `web_sales` por año (el de `ws_sold_timestamp`,
como en la consulta local) y, además, genera un pequeño informe de
qué host ha procesado cada partición. `mapPartitionsWithIndex` ejecuta una
función en cada partición y permite registrar allí
`socket.gethostname()`. Así se comprueba que YARN reparte el trabajo entre
varios DataNodes.

In [ ]:
from pathlib import Path

JOB_SCRIPT: str = r'''
# Trabajo independiente para YARN, lanzado con spark-submit --master yarn.
#
# No usa la SparkSession del notebook: es un proceso aparte, con su propio
# driver y su propia sesión.
import socket

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, year
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType

BUSINESS_OUTPUT = "/user/luser/s4/yarn-job/business-result"
EXECUTOR_REPORT_OUTPUT = "/user/luser/s4/yarn-job/executor-report"

spark = SparkSession.builder.appName("TCDM_S4_YARN_business").master("yarn").getOrCreate()

web_sales = spark.read.parquet("/datalake/raw/tpcds/web_sales")

sales_by_year = (
    web_sales.filter(col("ws_sold_timestamp").isNotNull())
    .groupBy(year("ws_sold_timestamp").alias("sold_year"))
    .agg(
        spark_sum("ws_net_profit").alias("beneficio_neto"),
        countDistinct("ws_order_number").alias("pedidos"),
        count("*").alias("lineas_de_venta"),
    )
    .orderBy("sold_year")
)
sales_by_year.coalesce(1).write.mode("overwrite").csv(BUSINESS_OUTPUT, header=True)

# Registra el host que ha ejecutado cada partición.
def describe_partition(partition_id, rows):
    rows = list(rows)
    yield (partition_id, socket.gethostname(), len(rows))


reports = (
    web_sales.select("ws_order_number").rdd.mapPartitionsWithIndex(describe_partition).collect()
)

report_schema = StructType(
    [
        StructField("partition", IntegerType(), False),
        StructField("executor_host", StringType(), False),
        StructField("records", LongType(), False),
    ]
)
(
    spark.createDataFrame(reports, report_schema)
    .orderBy("partition")
    .coalesce(1)
    .write.mode("overwrite")
    .csv(EXECUTOR_REPORT_OUTPUT, header=True)
)

distinct_hosts = {report[1] for report in reports}
print(f"Hosts remotos que procesaron alguna partición: {sorted(distinct_hosts)}")
spark.stop()
'''

Path("/tmp/s4_yarn_job.py").write_text(JOB_SCRIPT)
print("Escrito /tmp/s4_yarn_job.py")

### El envío con `spark-submit`

Los valores de memoria y núcleos son los mismos que se usaron al crear la sesión
sobre YARN, pasados ahora como opciones de la orden:
`--num-executors` y `--executor-cores`/`--executor-memory`, y el resto como
`--conf`. Hay una diferencia: aquí no hace falta `spark.yarn.isPython`, porque
`spark-submit` sabe que lanza un programa de Python. `spark.pyspark.driver.python`
apunta al Python de `/opt/tcdm/venv`, el único sitio donde hace falta el paquete
`pyspark` instalado.

Usamos el `spark-submit` de `/opt/tcdm/venv/bin`, el mismo entorno que el
kernel, para que coincida exactamente con la versión de PySpark que usa el resto
del notebook. La orden imprime muchas líneas de registro de Spark y puede tardar
unos minutos. La que importa está casi al final:
`Hosts remotos que procesaron alguna partición: [...]`.

In [ ]:
spark_submit_cmd: str = f"""/opt/tcdm/venv/bin/spark-submit \
    --master yarn \
    --deploy-mode client \
    --num-executors {requested_executors} \
    --executor-cores 1 \
    --executor-memory 512m \
    --conf spark.executor.memoryOverhead=256m \
    --conf spark.yarn.am.cores=1 \
    --conf spark.yarn.am.memory=256m \
    --conf spark.yarn.am.memoryOverhead=256m \
    --conf spark.scheduler.minRegisteredResourcesRatio=1.0 \
    --conf spark.scheduler.maxRegisteredResourcesWaitingTime=60s \
    --conf spark.pyspark.python=python3 \
    --conf spark.pyspark.driver.python=/opt/tcdm/venv/bin/python \
    /tmp/s4_yarn_job.py"""

print(spark_submit_cmd)

In [ ]:
!export HADOOP_CONF_DIR=/opt/bd/hadoop/etc/hadoop && {spark_submit_cmd}

### Leer lo que ha escrito el trabajo

El trabajo escribió en HDFS el resultado de negocio y el informe de hosts. Los
leemos y comprobamos lo mismo que en la sección anterior: que el resultado
coincide con el de la sesión local (`sales_by_year_local_rows`, que sólo existe
si «Beneficio neto por año» se ejecutó en este mismo kernel) y que al menos dos
DataNodes procesaron alguna partición.

In [ ]:
!hdfs dfs -cat /user/luser/s4/yarn-job/business-result/part-*.csv

In [ ]:
import csv
import io
from decimal import Decimal

yarn_csv: str = subprocess.run(
    ["hdfs", "dfs", "-cat", "/user/luser/s4/yarn-job/business-result/part-*.csv"],
    capture_output=True,
    text=True,
    check=True,
).stdout

yarn_by_year: dict[int, tuple[int, int, Decimal]] = {
    int(row["sold_year"]): (
        int(row["pedidos"]),
        int(row["lineas_de_venta"]),
        Decimal(row["beneficio_neto"]),
    )
    for row in csv.DictReader(io.StringIO(yarn_csv))
}
local_by_year: dict[int, tuple[int, int, Decimal]] = {
    row["sold_year"]: (
        row["pedidos"],
        row["lineas_de_venta"],
        row["beneficio_neto"],
    )
    for row in sales_by_year_local_rows
}

assert yarn_by_year == local_by_year, "El resultado de spark-submit no coincide con el local"
print(f"Local y spark-submit coinciden en los {len(local_by_year)} años.")

In [ ]:
!hdfs dfs -cat /user/luser/s4/yarn-job/executor-report/part-*.csv

In [ ]:
report_result: subprocess.CompletedProcess[str] = subprocess.run(
    ["bash", "-lc", "hdfs dfs -cat /user/luser/s4/yarn-job/executor-report/part-*.csv"],
    capture_output=True,
    text=True,
    check=True,
)
report_lines: list[str] = [line for line in report_result.stdout.splitlines()[1:] if line.strip()]
submit_hosts: set[str] = {line.split(",")[1] for line in report_lines}

print(f"Hosts que procesaron al menos una partición: {sorted(submit_hosts)}")
assert len(submit_hosts) >= 2, "Se esperan al menos dos hosts remotos distintos"